# App Trial: Marketing Mix Analysis

**Background**

This project analyzes the drivers of app trial volume using a marketing mix modeling approach. The dataset contains daily observations of media activity across multiple channels (TV, radio, search, and digital display) alongside app trial counts. The analysis focuses on understanding how different media channels contribute to user acquisition, how their effects persist over time, and whether interactions between channels enhance performance.

**Goal**

The main objective is to quantify the impact of each marketing channel on app trials and to identify potential synergies across channels. In particular, the project investigates:

- The direct contribution of each media channel
- The persistence of media effects over time (adstock)
- Diminishing returns to media investment (saturation)
- Cross-channel interactions, especially between TV and search
- The presence of second-screen effects

**Roadmap**

The analysis proceeds in several steps:

1. Data inspection and cleaning
2. Exploratory data analysis (EDA)
3. Feature engineering (adstock, saturation, seasonality)
4. Time-series modeling with autoregressive components
5. Model selection and diagnostics
6. Cross-channel interaction analysis
7. Second-screen effect investigation
8. Two-stage modeling to address endogeneity

> Note:
> - This notebook analyses the impact of marketing activities on daily App trials using a Marketing Mix Model (MMM) built on OLS regression.  Methodological details and robustness checks are documented throughout the notebook.
>
> - A separate Meridian (Bayesian MMM) notebook complements this analysis with unbiased channel attribution and budget optimization recommendations.

# 0. Load data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [ ]:
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec

In [ ]:
from scipy import stats

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_pacf, plot_acf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.anova import anova_lm
from statsmodels.tsa.stattools import adfuller
from statsmodels.iolib.summary2 import summary_col

from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit

In [ ]:
from IPython.display import display, Math, Latex, HTML

In [ ]:
import itertools

In [ ]:
data = pd.read_excel('IMA2026_AppTrials_DB.xlsx')
data.head()

## Shape, Missing Values and Data Types

We begin with a basic inspection of the dataset to understand its structure, identify any missing values, and verify that variable types are correct before proceeding with analysis.

In [ ]:
data.info()

In [ ]:
print("\nMissing values:")
missing = data.isna().sum()
print(missing[missing > 0])

Set date as index

In [ ]:
data = data.sort_values('Date').reset_index(drop=True)
data.set_index('Date', inplace=True)

In [ ]:
data.index

In [ ]:
data.columns

In [ ]:
data.rename(columns={'RADIO_Grp ': 'RADIO_Grp',
                     'WEB MENTIONS': 'WEB_MENTIONS',
                     'tot tv': 'tot_tv'},
            inplace=True)

Checking data ranges and gaps in the time series (missing days)

In [ ]:
print('Date range:', data.index.min(), 'to', data.index.max())

In [ ]:
full_range = pd.date_range(start=data.index.min(), end=data.index.max(), freq='D')
missing_dates = full_range.difference(data.index)
print("Missing dates:", missing_dates)

Check if a column is a sum of others

In [ ]:
data["sum_components"] = data["search_alwayson_spend"] + data["search_broad_spend"]
data["diff"] = data["SEARCH_spend"] - data["sum_components"]

print("Max absolute difference:", data["diff"].abs().max())
print("Rows where they don't match:", (data["diff"].abs() > 0.01).sum())

In [ ]:
data["sum_components"] = data["clic_search_alwayson"] + data["clic_search_broad"]
data["diff"] = data["CLIC_SEARCH"] - data["sum_components"]

print("Max absolute difference:", data["diff"].abs().max())
print("Rows where they don't match:", (data["diff"].abs() > 0.01).sum())

In [ ]:
TV_grp = [c for c in data.columns if 'TV' in c]
TV_grp

In [ ]:
TV_grp.remove('TV_Grp_Rai')
TV_grp

In [ ]:
data["sum_components"] = data[TV_grp].sum(axis=1)
data["diff"] = data["tot_tv"] - data["sum_components"]

print("Max absolute difference:", data["diff"].abs().max())
print("Rows where they don't match:", (data["diff"].abs() > 0.01).sum())


In [ ]:
data.drop(columns=["sum_components", "diff"], inplace=True)

In [ ]:
num_col = data.columns.tolist()
num_col.remove('DUMMY_SEARCH')

# 1. EDA

## 1.1. Time Series

### Trials over time

In [ ]:
# Find where broad search turns on
broad_start = data.index[data['DUMMY_SEARCH'] == 1][0]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(data.index, data['Trial'], linewidth=1.2, color='steelblue', zorder=3, label='Trials')
ax.axvspan(broad_start, data.index[-1], alpha=0.2, color='lightgreen', zorder=1, label='Broad search active')

rug_channels = {
    'tot_tv':       ('darkred',             0),
    'RADIO_Grp':    ('turquoise',      1),
    'WEB_MENTIONS': ('gray',          2),
    'CPD_spend':    ('darkorange',    3),
}

ymin, ymax = data['Trial'].min(), data['Trial'].max()
n_rugs     = len(rug_channels)
rug_gap    = (ymax - ymin) * 0.04
rug_height = (ymax - ymin) * 0.025
rug_zone   = n_rugs * (rug_height + 2) + rug_gap

ax.set_ylim(ymin - rug_zone, ymax * 1.05)

for col, (color, i) in rug_channels.items():
    active = data.index[data[col] > 0]
    y_base = ymin - rug_gap - (i + 1) * rug_height - i * 2
    ax.vlines(active, y_base, y_base + rug_height,
              color=color, linewidth=1.2, alpha=0.7, zorder=2, label=col)

ax.set_title("Trials Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Trials")
ax.legend(ncol=3, loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Daily app trials plotted over the full observation window (May–December 2016), with media activity indicators overlaid. The shaded area marks the activation of the broad search campaign in October 2016.

The series shows a notable spike in trial volume in November, coinciding with a surge in TV, Radio, and CPD activity — before settling back to a moderately elevated level through December. The pre-November period is comparatively flat and volatile, with limited offline
media presence.

The considerable day-to-day variability motivates a regression-based marketing mix framework -
trials are unlikely to be driven by a single channel or a simple deterministic pattern.

### Online channels

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

spend_cols = [c for c in data.columns if "spend" in c.lower()
              and c not in ('search_alwayson_spend', 'search_broad_spend')]
click_cols = [c for c in data.columns if "clic" in c.lower()
              and c not in ('clic_search_alwayson', 'clic_search_broad')]

for col in spend_cols:
    if col.startswith('CPC'):
      color = 'mediumpurple'
    elif col.startswith('CPD'):
      color = 'darkorange'
    else:
      color = 'mediumseagreen'
    ax1.plot(data.index, data[col], label=col, linewidth=1.2, alpha=0.8, color = color)
ax1.set_title("Spend on Online Channels Over Time")
ax1.set_ylabel("Spend")
ax1.legend(ncol=3, loc="upper right")

for col in click_cols:
    if col.startswith('CPC'):
      color = 'mediumpurple'
    elif col.startswith('CPD'):
      color = 'darkorange'
    else:
      color = 'seagreen'
    ax2.plot(data.index, data[col], label=col, linewidth=1.2, alpha=0.8, linestyle='--', color = color)
ax2.set_title("Clicks per Online Channel Over Time")
ax2.set_ylabel("Clicks")
ax2.set_xlabel("Date")
ax2.legend(ncol=3, loc="upper right")

plt.tight_layout()
plt.show()


Regarding spend,
- CPC/CPM dominates spend in the early period, with a sharp spike in late June before declining.

- CPD and Search spend remain low throughout but become more active from September onward.

The clicks picture is strikingly different:
- CPD generates by far the most clicks from September onwards, despite modest spend - suggesting a very low CPD cost per click.
- CPC/CPM clicks remain flat and low throughout, hinting at a display/awareness rather than direct-response role.

In [ ]:
search_spend_cols = ['search_alwayson_spend', 'search_broad_spend']
search_click_cols = ['clic_search_alwayson', 'clic_search_broad']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for col in search_spend_cols:
    if 'broad' in col:
        ax1.plot(data.index[data.index >= broad_start],
                 data.loc[data.index >= broad_start, col],
                 label=col, linewidth=1, zorder=2, color = 'lightgreen')
    else:
        ax1.plot(data.index, data[col], label=col, linewidth=1, alpha=0.8, zorder=2, color = 'darkgreen')

for col in search_click_cols:
    if 'broad' in col:
        ax2.plot(data.index[data.index >= broad_start],
                 data.loc[data.index >= broad_start, col],
                 label=col, linewidth=1, linestyle='--', zorder=2, color = 'lightgreen')
    else:
        ax2.plot(data.index, data[col], label=col, linewidth=1, alpha=0.8, linestyle='--', zorder=2, color = 'darkgreen')

ax1.set_ylabel("Spend")
ax1.set_title("Search Over Time")
ax1.set_ylim(bottom=0)
ax1.legend()

ax2.set_ylabel("Clicks")
ax2.set_xlabel("Date")
ax2.set_ylim(bottom=0)
ax2.legend()

# Shade the broad-active regime on both panels
for ax in (ax1, ax2):
    ax.axvspan(broad_start, data.index[-1], alpha=0.1, color='lightgreen', zorder = 3)

# Single legend entry for the shading — attach to bottom panel
ax2.axvspan(broad_start, broad_start, alpha=0.1, color='lightgreen',
            label='Broad search active (DUMMY_SEARCH = 1)', zorder = 3)
ax2.legend()

plt.tight_layout()
plt.show()

- Always-on search runs continuously at a relatively stable spend level, with a step-up in activity post-October coinciding with the broad campaign launch.
- Broad search spend and clicks are zero before October, confirming the regime shift captured by `DUMMY_SEARCH`.

Post-launch, always-on continues to drive the majority of search clicks, while broad search contributes a secondary, more volatile stream.

The two components are sufficiently distinct to warrant separate treatment in the model.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(data.index, data['WEB_MENTIONS'], alpha=0.5, color = 'gray')
plt.title("Web mentions Over Time")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()

Web mentions serve as a proxy for organic buzz or PR activity. Unlike paid channels, this variable is not directly controlled by the advertiser, and its values are discrete and sparse (integer counts, mostly 0–5). We include it as
a potential control variable rather than a primary media driver.

### Offline channels

In [ ]:
tv_cols   = [c for c in data.columns if "TV_Grp" in c]
fig, ax = plt.subplots(figsize=(12, 4))
colors = ['rosybrown', 'lightcoral', 'red', 'maroon']
for col in tv_cols:
    if col == 'TV_Grp_Mediaset_Premium_Sky':
        ax.plot(data.index, data[col], label=col, linewidth=1.2, linestyle = '--', color = 'mediumvioletred')
    elif col != 'TV_Grp_Rai':
        ax.plot(data.index, data[col], label=col, linewidth=1.2, color = colors[int(col[-1])-1])

ax.set_title(f"Gross Rating Points (GRPs) Over Time")
ax.set_ylim(bottom=0)
ax.legend(ncol=3, loc="upper left", fontsize = 8)
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4))

ax1.plot(data.index, data['RADIO_Grp'], label='Radio',
         linewidth=1.2, alpha=0.8, linestyle='--', color='turquoise')
ax1.set_ylabel("Radio GRPs", color='turquoise')
ax1.tick_params(axis='y', labelcolor='turquoise')

ax2 = ax1.twinx()
ax2.plot(data.index, data['tot_tv'], label='TV',
         linewidth=1.2, alpha=0.8, color='darkred')
ax2.set_ylabel("TV GRPs", color='darkred')
ax2.tick_params(axis='y', labelcolor='darkred')

# Merge legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.set_xlabel("Date")

# hide grids for both axes
ax1.grid(False)
ax2.grid(False)

plt.title("TV vs Radio GRPs Over Time")
plt.tight_layout()
plt.show()

Television activity is measured in **Gross Rating Points (GRPs)** across multiple broadcasters (RAI flights and Mediaset/Premium/Sky). Radio GRPs are plotted alongside total TV GRPs on a dual axis to compare the timing and
intensity of offline media investment.

Two observations stand out:
- TV and Radio activity is concentrated in **November–December 2016**, with near-zero presence beforehand.
- The sparse, burst-like pattern of TV and Radio has direct implications for modelling: saturation curves cannot be reliably identified from a handful of non-zero observations.

**OVERALL**

The media time-series plots show that not all channels behave in the same way.

Some channels appear to be more continuously active, while others are concentrated in bursts or sparse periods. This matters because channels with very limited non-zero observations may be harder to estimate precisely in regression models.

## 1.2 Distribution and outliers

**Non-zero presence**

In [ ]:
candidate_cols = [
    'CPC/CPM_spend', 'CPD_spend',
    'search_alwayson_spend', 'search_broad_spend',
    'TV_Grp_Mediaset_Premium_Sky', 'TV_Grp_Rai',
    'TV_Grp_Rai3_OnAir1', 'TV_Grp_Rai3_OnAir2',
    'TV_Grp_Rai3_OnAir3', 'TV_Grp_Rai3_OnAir4',
    'RADIO_Grp', 'WEB_MENTIONS']

non_zero_days = pd.DataFrame({
    'variable': candidate_cols,
    'non_zero_days': [(data[c] > 0).sum() for c in candidate_cols],
    'pct_non_zero': [round(100 * (data[c] > 0).mean(), 1) for c in candidate_cols]
}).sort_values('non_zero_days', ascending=False)

In [ ]:
def get_color(var):
    var_lower = var.lower()
    if 'cpc' in var_lower or 'cpm' in var_lower:
        return 'mediumpurple'
    elif 'search' in var_lower:
        return 'mediumseagreen'
    elif 'web' in var_lower:
        return 'gray'
    elif 'tv' in var_lower:
        return 'darkred'
    elif 'cpd' in var_lower:
        return 'darkorange'
    elif 'radio' in var_lower:
        return 'turquoise'
    elif 'trial' in var_lower:
        return 'steelblue'

In [ ]:
plot_df = non_zero_days.sort_values('pct_non_zero', ascending=True)
colors = [get_color(v) for v in plot_df['variable']]

plt.figure(figsize=(10, 6))

for i, (_, row) in enumerate(plot_df.iterrows()):
    c = get_color(row['variable'])
    plt.hlines(y=row['variable'], xmin=0, xmax=row['pct_non_zero'], color=c)
    plt.plot(row['pct_non_zero'], row['variable'], 'o', color=c)
    plt.text(row['pct_non_zero'] + 1, row['variable'], f"{row['pct_non_zero']:.1f}%", va='center', fontsize=9)

plt.xlabel('% of days with non-zero values')
plt.ylabel('')
plt.title('Non-Zero Presence of Media Variables', fontweight='bold')
plt.xlim(0, 110)
plt.tight_layout()
plt.show()

This shows how often each media variable is active over the sample period.

The purpose of this check is to evaluate whether some channels are too sparse to be modeled reliably. If a variable is non-zero only on very few days, its coefficient may become unstable and difficult to interpret.

**Distribution**

In [ ]:
broad_mask = data['DUMMY_SEARCH'] == 1
cpd_mask = data['CPD_spend'] > 0

hist_config = {
    'Trial'                : (data['Trial'], 'Trial'),
    'WEB_MENTIONS'         : (data['WEB_MENTIONS'], 'WEB_MENTIONS'),
    'CPC/CPM_clic'         : (data['CPC/CPM_clic'], 'CPC/CPM_clic'),
    'CPC/CPM_spend'        : (data['CPC/CPM_spend'], 'CPC/CPM_spend'),
    'CPD_CLIC'             : (data.loc[cpd_mask, 'CPD_CLIC'], 'CPD_CLIC\n(active days only)'),
    'CPD_spend'            : (data.loc[cpd_mask, 'CPD_spend'], 'CPD_spend\n(active days only)'),
    'CLIC_SEARCH'          : (data['CLIC_SEARCH'], 'CLIC_SEARCH'),
    'SEARCH_spend'         : (data['SEARCH_spend'], 'SEARCH_spend'),
    'clic_search_alwayson' : (data['clic_search_alwayson'], 'clic_search_alwayson'),
    'search_alwayson_spend': (data['search_alwayson_spend'], 'search_alwayson_spend'),
    'clic_search_broad'    : (data.loc[broad_mask, 'clic_search_broad'],
                              'clic_search_broad\n(campaign period only)'),
    'search_broad_spend'   : (data.loc[broad_mask, 'search_broad_spend'],
                              'search_broad_spend\n(campaign period only)'),
}

n_cols = 4
n_rows = -(-len(hist_config) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3))

for ax, (series, title) in zip(axes.flatten(), hist_config.values()):
    series.hist(ax=ax, bins=30, color=get_color(title), edgecolor='white')
    ax.set_title(title, fontsize=8)

for ax in axes.flatten()[len(hist_config):]:
    ax.set_visible(False)

plt.suptitle('Distribution of Raw Input Variables',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

Right-skewed distributions and excess zeros are common in media data and inform the transformations applied in the feature engineering step (adstock, saturation, log transformation of the dependent variable).

## 1.3. Correlation

In [ ]:
corr = data[num_col].corr()
n = len(num_col)

fig, ax = plt.subplots(figsize=(max(10, n * 0.7), max(8, n * 0.6)))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="Spectral",
    center=0, linewidths=0.4, linecolor="white",
    annot_kws={"size": 7}, ax=ax
)
ax.set_title("Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

Several patterns are immediately visible:

- Search variables (`CLIC_SEARCH`, `clic_search_alwayson`, `SEARCH_spend`) show the strongest positive correlations with `Trial`.
- CPC/CPM metrics are **negatively** correlated with most other channels, reflecting that the CPC/CPM campaign was most active early in the period when trials were lower.
- Several channel pairs show correlations above 0.85 (e.g. spend and clicks within the same channel), suggesting that using both in the same model would introduce redundancy. One representative variable per channel will be selected in the modelling step.

## 1.4. SPEND vs CLICKS — paired channel check

In [ ]:
spend_clic_pairs = [
    ("CPC/CPM_spend", "CPC/CPM_clic"),
    ("CPD_spend",     "CPD_CLIC"),
    ("SEARCH_spend",  "CLIC_SEARCH"),
    ("search_alwayson_spend", "clic_search_alwayson"),
    ("search_broad_spend",    "clic_search_broad"),
]
available_pairs = [(s, c) for s, c in spend_clic_pairs if s in data.columns and c in data.columns]

fig, axes = plt.subplots(2, 3, figsize=(12, 6))

for ax, (spend, clic) in zip(axes.flatten()[:-1], available_pairs):
    sns.scatterplot(data=data, x=spend, y=clic, hue='Trial', palette= 'summer',
                    alpha=0.5, size='Trial', ax=ax, legend= (ax==axes.flatten()[0]))
    ax.set_xlabel(spend, fontsize=8)
    ax.set_ylabel(clic, fontsize=8)
    ax.set_title(f"{clic}\nvs {spend}")


# Place legend in the 6th slot
handles, labels = axes.flatten()[0].get_legend_handles_labels()
axes.flatten()[0].get_legend().remove()  # remove the one on axes[0]

axes.flatten()[5].legend(
    handles, labels,
    title='Trial',
    loc='center',
    frameon=False  # optional: no box border
)
axes.flatten()[5].axis('off')
axes.flatten()[5].set_facecolor('none')
axes.flatten()[5].set_visible(True)  # need it visible to render the legend
fig.suptitle("Spend vs Clicks — Channel Pairs")

plt.tight_layout()
plt.show()

The scatter plots above show the relationship between spend and clicks for each
digital channel, with point size and colour encoding daily trial volume.

- **CPC/CPM**: spend and clicks move together tightly in a near-linear relationship.

    The two metrics are essentially interchangeable for this channel.

- **CPD**: the same spend level produces wildly different click volumes across days, reflecting erratic buying behaviour rather than a stable cost-per-click.

    Despite generating high click volumes on active days, CPD clicks show limited correlation with trials, suggesting a high-intent gap typical of display formats: clicks do not reliably translate into app trials.

- **Search (total)** — a clear two-regime structure is visible: a low-activity cluster before October 2016 (pre-broad-search launch) and a higher-activity regime after.
    
    Within each regime, spend and clicks track each other closely.

    - **search_alwayson**: the tightest relationship of all channels.
    
        Always-on campaigns ran continuously with a stable cost-per-click, confirming this is the backbone of search activity.

    - **search_broad**: loose and erratic.
    
        Several days show non-zero spend but near-zero clicks, indicating poor buying efficiency. This, combined with a median of zero clicks across the full sample, makes broad an unreliable standalone predictor.

## 1.5 TV and Search: a preview of cross-channel dynamics

The correlation matrix already hints at a notable relationship between # TV activity and search behaviour.

To make this more concrete, the scatter plot below isolates the **total TV GRPs vs search clicks** relationship, and the time-series overlay shows how the two series move together.

These patterns will be investigated more rigorously in below section on Second-Screen Effects, where we test whether TV exposure causally drives search activity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
axes[0].scatter(data['tot_tv'], data['CLIC_SEARCH'], alpha=0.5, s=20)
axes[0].set_xlabel('Total TV GRPs')
axes[0].set_ylabel('Search clicks (CLIC_SEARCH)')
axes[0].set_title('TV pressure vs search activity')
axes[0].grid(alpha=0.2)

# Time-series overlay
ax1 = axes[1]
ax1.plot(data.index, data['CLIC_SEARCH'], label='Search clicks', linewidth=1.2, color='mediumseagreen')
ax1.set_ylabel('CLIC_SEARCH', color='mediumseagreen')
ax2 = ax1.twinx()
ax2.bar(data.index, data['tot_tv'], alpha=0.25, width=1.0, color='firebrick', label='Total TV')
ax2.set_ylabel('Total TV GRPs', color='firebrick')
ax2.tick_params(axis='y', labelcolor='firebrick')
ax2.grid(False)
axes[1].set_title('Search clicks rise during TV flights')

fig.tight_layout()
plt.show()

# 2. Feature engineer


## 2.1. Adstock

The standard approach is **geometric decay**:

$$\text{Adstock}_t = x_t + \alpha \cdot \text{Adstock}_{t-1}$$

where:
- $x_t$ = raw media value (GRPs, spend, clicks) at time $t$
- $\alpha \in [0, 1]$ = **decay rate** (the "memory" parameter)
  - $\alpha = 0$ → no carryover (instantaneous effect only)
  - $\alpha = 0.5$ → 50% of yesterday's stock carries over
  - $\alpha = 0.9$ → very slow decay, effect persists for weeks

In practice, $\alpha$ is either set from industry benchmarks or estimated by scanning multiple values and checking model fit (or using Bayesian MMM tools like Robyn/Meridian)

In [ ]:
def geometric_adstock(series: pd.Series, decay: float) -> pd.Series:
    """
    Parameters:
    - series : pd.Series  — raw media values (GRPs, spend, clicks)
    - decay  : float      — λ in [0, 1]; higher = longer memory

    Returns:  pd.Series  — adstocked (transformed) media values
    """
    adstocked = np.zeros(len(series))
    for t, val in enumerate(series):
        if t == 0:
            adstocked[t] = val
        else:
            adstocked[t] = val + decay * adstocked[t - 1]
    return pd.Series(adstocked, index=series.index)

### TV & Radio

In [ ]:
# ── Define decay rates per channel ────────────────────────────────────────────
DECAY_TV    = 0.7   # TV GRPs decay slowly
DECAY_RADIO = 0.5   # Radio decays faster


# ── Apply adstock to each media variable ──────────────────────────────────────
# Paid/Commercial TV (Mediaset, Premium, Sky)
data['TV_Paid_adstock'] = geometric_adstock(data['TV_Grp_Mediaset_Premium_Sky'], DECAY_TV)

# Public TV (RAI) — all OnAir flights combined
data['TV_RAI_adstock']  = geometric_adstock(data['TV_Grp_Rai'], DECAY_TV)
data['TV_RAI_adstock1']  = geometric_adstock(data['TV_Grp_Rai3_OnAir1'], DECAY_TV)
data['TV_RAI_adstock2']  = geometric_adstock(data['TV_Grp_Rai3_OnAir2'], DECAY_TV)
data['TV_RAI_adstock3']  = geometric_adstock(data['TV_Grp_Rai3_OnAir3'], DECAY_TV)
data['TV_RAI_adstock4']  = geometric_adstock(data['TV_Grp_Rai3_OnAir4'], DECAY_TV)

# Radio
data['Radio_adstock']   = geometric_adstock(data['RADIO_Grp'], DECAY_RADIO)

### Search

For all digital channels (**Search**, **CPC/CPM**, **CPD**), spend is used as the input to the adstock transformation rather than clicks.

- Spend is the direct measure of media investment, making elasticity interpretable without additional assumptions
- CPC/CPM mixes two buying models (cost-per-click and cost-per-mille); spend is the only consistent unit across both

> **Note on CPD:** preliminary EDA suggested weak correlation between CPD spend and trials. Adstock is constructed here regardless — whether the channel has any explanatory power will be assessed in the modelling step.

In [ ]:
# Search (total clicks as media pressure proxy)
DECAY_SEARCH = 0.2  # Search is near-instantaneous

search_total           = data['SEARCH_spend']
data['Search_adstock'] = geometric_adstock(search_total, DECAY_SEARCH)

In [ ]:
data['SearchAO_adstock'] = geometric_adstock(data['search_alwayson_spend'], DECAY_SEARCH)
data['SearchBroad_adstock'] = geometric_adstock(data['search_broad_spend'], DECAY_SEARCH)

### CPC/CPM & CPD


In [ ]:
DECAY_CPD = 0.1    # performance channel — pay per download, near-instantaneous
DECAY_CPC = 0.15   # same logic — click-based, mostly instantaneous

data['CPD_adstock'] = geometric_adstock(data['CPD_spend'], decay=DECAY_CPD)
#data['CPD_sat']     = log_saturation(data['CPD_adstock'])
data['CPC_adstock'] = geometric_adstock(data['CPC/CPM_spend'], decay=DECAY_CPC)

In [ ]:
search_alwayson_total = data['search_alwayson_spend']
search_broad_total = data['search_broad_spend']

data['SearchAO_adstock'] = geometric_adstock(search_alwayson_total, DECAY_SEARCH)
data['SearchBroad_adstock'] = geometric_adstock(search_broad_total, DECAY_SEARCH)

## 2.2. Diminishing Returns (Saturation)

The purpose of saturation is to capture diminishing returns — beyond a certain level of
media pressure, additional spend or exposure generates a smaller incremental effect on
app trials.

We use an **arctangent function** as the saturation curve:

$$\text{AAdstock}_t = \arctan\!\left(\frac{\text{Adstock}_t}{\beta}\right), \quad 0 < \beta < \frac{\max(\text{Adstock})}{2}$$

This function is well-suited for MMM because it increases steeply at low media pressure,
flattens gradually as intensity grows, and is bounded — reflecting a natural ceiling on
media effectiveness. By applying it on top of adstock, the transformed variable captures
both **persistence over time** (adstock) and a **nonlinear response to media intensity**
(saturation).

Saturation only makes sense when a channel has **continuous, non-zero variation** - otherwise the curve cannot be identified from the data. TV and Radio are sparse (mostly zeros, with a few spikes in Nov–Dec), so we apply saturation selectively:

| Channel | Transformation | Reason |
|---|---|---|
| TV RAI OnAir 1–4 | Adstock only | Sparse, flight-based spikes — curve not identifiable |
| TV Paid (Mediaset/Sky) | Adstock only | Same |
| Radio | Adstock only | Nearly all zeros |
| Search | Adstock → saturation | High volume, continuous daily variation |
| CPC/CPM | Adstock → saturation | Continuous spend throughout the period |
| CPD | Adstock → saturation | Active from Sep 2016; significance tested in model |

Note: For online channels, we use a low decay rate, meaning most of the adstock depletes each day and the transformed value is close to raw spend. The adstock step is retained for methodological consistency across all channels before saturation is applied.

**Beta selection**: $\beta$ controls the inflection point of the curve - smaller $\beta$ compresses
the input range and implies saturation kicks in earlier. $\beta$ is selected via grid search
over `beta_pct_grid` (fractions of the channel's max adstock), keeping only configurations
where all media coefficients remain positive — a business constraint ensuring no channel
is estimated to harm trials.

In [ ]:
def atan_saturation(series: pd.Series, beta: float) -> pd.Series:
    return pd.Series(np.arctan(series.values / beta), index=series.index)

In [ ]:
# Initial saturation (default beta = series median — used for EDA only).
for raw_col, sat_col in [("Search_adstock",  "Search_sat"),
                         ("CPD_adstock",     "CPD_sat"),
                         ("CPC_adstock",     "CPC_sat"),
                         ("SearchAO_adstock", "SearchAO_sat"),
                         ("SearchBroad_adstock", "SearchBroad_sat")]:
    beta_init = max(float(data[raw_col].median()), 1e-3)
    data[sat_col] = atan_saturation(data[raw_col], beta_init)


In [ ]:
data.columns

In [ ]:
sat_cols = ['Search_sat', 'CPC_sat', 'CPD_sat', 'SearchAO_sat', 'SearchBroad_sat']

print(f"Saturation applied to {len(sat_cols)} features: {sat_cols}\n")
print(data[sat_cols].describe().round(4))

# Bounds check: all values should be in [0, π/2]
assert data[sat_cols].min().min() >= 0,          "ERROR: values below 0"
assert data[sat_cols].max().max() <= np.pi / 2,  "ERROR: values above π/2"
print(f"\nAll values bounded between 0 and π/2 ({np.pi/2:.4f})")
print(f"No missing values — {data[sat_cols].notna().all().all()}")


## 2.3. Seasonal Dummies

To account for weekly seasonality, we create day-of-week dummy variables based on the date index, dropping one category to avoid perfect multicollinearity. The included dummies therefore measure deviations relative to the omitted reference day, and are treated here as control variables rather than main drivers of app trials.

App trials are not uniform over time. They follow recurring calendar patterns unrelated to advertising activity. If left uncontrolled, the model may falsely attribute seasonal spikes to whichever media happened to air at that time, confounding our estimates of media effectiveness.

- Day-of-week: people are more likely to download apps on certain days
- Month / Quarter: new year resolutions, summer slumps
- Italian public holidays: Ferragosto, Natale

However, our data covers only a single year (May–Dec 2016), meaning month and holiday dummies would be highly collinear with campaign timing and risk absorbing the very media effects we aim to identify. We therefore limit seasonal controls to day-of-week dummies only.

In [ ]:
# ── Extract time components ────────────────────────────────────────────────────
data['day_of_week'] = data.index.dayofweek   # 0 = Monday, 6 = Sunday
data['day_name']    = data.index.day_name()

# ── Day-of-week dummies (drop Monday as reference) ────────────────────────────
dow_dummies = pd.get_dummies(data['day_name'], prefix='dow', drop_first=False)
dow_dummies = dow_dummies.drop(columns=['dow_Monday'], errors='ignore')  # reference = Monday
df = pd.concat([data, dow_dummies], axis=1)

print('Day-of-week dummies:', [c for c in df.columns if c.startswith('dow_')])

In [ ]:
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(8,5))
sns.boxplot(data=data,
            x='day_name', y='Trial',
            order=dow_order,
            showmeans=True,
            meanprops={'marker': 'D',
                       'markerfacecolor': 'gold',
                       'markeredgecolor': 'white',
                       'markersize': 6})

mean_legend = Line2D(
    [0], [0],
    marker='D',
    color='w',
    label='Day mean',
    markerfacecolor='gold',
    markeredgecolor='black',
    markersize=8
)
plt.axhline(data['Trial'].mean(), color='gold', linestyle='--', linewidth=1, label='Overall mean')
plt.legend(handles=[mean_legend], loc='upper left')

plt.title('App Trials by Day of Week', fontweight='bold')
plt.xlabel('')
plt.ylabel('Daily Trials')
plt.xticks(fontsize=9)
plt.show()

We should use separate day-of-week dummies rather than a single weekend binary. From the plotA, Sunday is the highest-trial day (\~120 avg) while Saturday is the lowest (\~98) — collapsing them into one dummy `weekend` would cause the effects to cancel out, losing real signal.

## 2.4 Dynamic Features: Log Transformation and Lagged Trials

### Log Transformation

Raw `Trial` is right-skewed, which violates the OLS assumption of homoscedastic errors. We apply a log transformation to stabilise variance and produce better-behaved residuals.

`log_Trial` serves as the dependent variable in all models.

In [ ]:
data['log_Trial'] = np.log(data['Trial'])

### Lag Order Selection (PACF)

Daily app downloads are not independent across time - today's trials are partly driven by yesterday's, through word-of-mouth, app store momentum, and user referrals. Ignoring this autocorrelation structure would bias media coefficients and invalidate standard errors.

We run the Partial Autocorrelation Function (PACF) on `log_Trial` to determine the appropriate lag order. The PACF isolates the direct relationship between the series and each of its own lags, stripping out indirect effects that propagate through intermediate lags.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_pacf(data['log_Trial'], lags=14, ax=ax)
ax.set_title('PACF of log_Trial — significant spikes at lags 1 and 2 → AR(2) baseline')
ax.set_xlabel('Lag (days)')
plt.tight_layout()
plt.show()


The PACF shows significant spikes at lags 1 and 2, with nothing meaningful beyond - the standard signature of an AR(2) process. Yesterday's and the day before's downloads carry predictive power, but older history does not.

We therefore include `log_Trial_lag1` and `log_Trial_lag2` as regressors in all model specifications.

In [ ]:
data['log_Trial_lag1'] = data['log_Trial'].shift(1)
data['log_Trial_lag2'] = data['log_Trial'].shift(2)
data_l2 = data.dropna().copy()

# 3. Unit Root / Stationarity Testing

Before estimating any model, we confirm that `log_Trial` is stationary. A unit root in the dependent variable would render OLS coefficients spurious.

We run the Augmented Dickey-Fuller (ADF) test with lag order selected automatically
by AIC. We test two series:

- **Trial (levels)** — baseline check on the raw outcome variable
- **log_Trial** — the transformed series that enters the model

The null hypothesis is the presence of a unit root. Rejection at p < 0.05 confirms
stationarity and validates the use of OLS.

In [ ]:
def adf_test(series, name):
    result = adfuller(series.dropna(), autolag='AIC')
    return {
        'Series'      : name,
        'ADF Stat'    : round(result[0], 4),
        'p-value'     : round(result[1], 4),
        'Used Lags'   : result[2],
        'Obs'         : result[3],
        'CV 1%'       : round(result[4]['1%'], 3),
        'CV 5%'       : round(result[4]['5%'], 3),
        'CV 10%'      : round(result[4]['10%'], 3),
        'Stationary?' : '✅ Yes' if result[1] < 0.05 else '❌ No'
    }

adf_results = pd.DataFrame([
    adf_test(data['Trial'],     'Trial (levels)'),
    adf_test(data['log_Trial'], 'log_Trial'),
])

adf_results

Both series reject the unit root null at the 5% level, confirming stationarity.

AIC selected different lag lengths for the two series, which is expected: the log transformation reduces autocorrelation, requiring fewer lags to whiten the residuals.

The lag count for `log_Trial` aligns with the AR(2) structure identified from the PACF, providing independent confirmation of the lag order choice.

The sample contains a known regime shift in October 2016, captured by `DUMMY_SEARCH`. ADF has reduced power under structural breaks; however, both series reject the null comfortably, so the break does not affect the stationarity conclusion.

# 4. Models

## 4.1. Models

### Pure AR(2)
We start with the simplest possible model: trial momentum only, no controls, no media.

log_Trial ~ log_Trial_lag1 + log_Trial_lag2

In [ ]:
formula_base1 = '''log_Trial ~ log_Trial_lag1 + log_Trial_lag2'''

model_base1 = smf.ols(formula_base1, data=data_l2).fit()

### AR(2) + Dummy search + Day-of-week

Two sources of non-media variation need to be controlled for before
introducing any advertising variables.

- **DUMMY_SEARCH** captures a permanent level shift in October 2016 when search advertising switched on and remained active. This is a regime change in the series — without controlling for it, any media variable active in that
period risks absorbing the shift.

- **Day-of-week dummies** capture calendar patterns unrelated to advertising.

    As seen in the EDA, Sunday is notably higher than the overall mean while Saturday is below — opposite directions, which is why we use separate dummies rather than a weekend binary. Monday is the reference level.
```
log_Trial ~ log_Trial_lag1 + log_Trial_lag2 + DUMMY_SEARCH + C(day_name)
```

In [ ]:
formula_base2 = '''log_Trial ~ log_Trial_lag1 + log_Trial_lag2 + DUMMY_SEARCH + C(day_name)'''

model_base2 = smf.ols(formula_base2, data=data_l2).fit(cov_type="HAC",cov_kwds={"maxlags": 7})

In [ ]:
formula_base3 = '''log_Trial ~ log_Trial_lag1 + log_Trial_lag2 + DUMMY_SEARCH + C(day_name) + WEB_MENTIONS'''

model_base3 = smf.ols(formula_base3, data=data_l2).fit(cov_type="HAC",cov_kwds={"maxlags": 7})

### Adding TV adstock

In [ ]:
formula_tv = '''log_Trial ~ log_Trial_lag1 + log_Trial_lag2
              + DUMMY_SEARCH + C(day_name)
              + TV_RAI_adstock1 + TV_RAI_adstock2 + TV_RAI_adstock3 + TV_RAI_adstock4
              + TV_Paid_adstock + Radio_adstock'''

model_tv = smf.ols(formula_tv, data=data_l2).fit(cov_type='HAC', cov_kwds={'maxlags': 7})

In [ ]:
formula_tv2 = '''log_Trial ~ log_Trial_lag1 + log_Trial_lag2
              + DUMMY_SEARCH + C(day_name)
              + TV_RAI_adstock + TV_Paid_adstock + Radio_adstock'''

model_tv2 = smf.ols(formula_tv2, data=data_l2).fit(cov_type='HAC', cov_kwds={'maxlags': 7})

### Adding online channels

In [ ]:
formula_all1 = '''log_Trial ~ log_Trial_lag1 + log_Trial_lag2
              + C(day_name)
              + TV_RAI_adstock1 + TV_RAI_adstock2 + TV_RAI_adstock3 + TV_RAI_adstock4
              + TV_Paid_adstock + Radio_adstock
              + Search_sat
              + CPC_sat + CPD_sat'''

model_all1 = smf.ols(formula_all1, data=data_l2).fit(cov_type='HAC', cov_kwds={'maxlags': 7})

In [ ]:
formula_all2 = '''log_Trial ~ log_Trial_lag1 + log_Trial_lag2
              + DUMMY_SEARCH + C(day_name)
              + TV_RAI_adstock1 + TV_RAI_adstock2 + TV_RAI_adstock3 + TV_RAI_adstock4
              + TV_Paid_adstock + Radio_adstock
              + Search_sat
              + CPC_sat + CPD_sat'''

model_all2 = smf.ols(formula_all2, data=data_l2).fit(cov_type='HAC', cov_kwds={'maxlags': 7})

In [ ]:
formula_all3 = '''log_Trial ~ log_Trial_lag1 + log_Trial_lag2
              + DUMMY_SEARCH + C(day_name)
              + TV_RAI_adstock
              + TV_Paid_adstock + Radio_adstock
              + Search_sat
              + CPC_sat + CPD_sat'''

model_all3 = smf.ols(formula_all3, data=data_l2).fit(cov_type='HAC', cov_kwds={'maxlags': 7})

In [ ]:
formula_all4 = '''log_Trial ~ log_Trial_lag1 + log_Trial_lag2
              + C(day_name)
              + TV_RAI_adstock1 + TV_RAI_adstock2 + TV_RAI_adstock3 + TV_RAI_adstock4
              + TV_Paid_adstock + Radio_adstock
              + SearchAO_sat + SearchBroad_sat
              + CPC_sat + CPD_sat'''

model_all4 = smf.ols(formula_all4, data=data_l2).fit(cov_type='HAC', cov_kwds={'maxlags': 7})

In [ ]:
# Generate the summary table comparing models
summary = summary_col(
    [model_base1, model_base2, model_base3, model_tv, model_tv2,
     model_all1,  model_all2, model_all3, model_all4],
    stars=True,
    float_format="%.3f",
    model_names=['base 1', 'base2', 'base3', 'base + tv', 'base + tv2',
                 'all channels1', 'all channels2', 'all channels3', 'all channels4'])

df = summary.tables[0]

# Drop rows where the index is empty (std error rows)
df_clean = df[df.index.str.strip() != '']

# Add AIC, BIC, Adj R², N rows manually
models = [model_base1, model_base2, model_base3, model_tv, model_tv2,
          model_all1,  model_all2, model_all3, model_all4]
names  = ['base 1', 'base2', 'base3', 'base + tv', 'base + tv2',
          'all channels1', 'all channels2', 'all channels3', 'all channels4']

extra_rows = {'AIC': [f"{m.aic:.2f}" for m in models],
              'BIC': [f"{m.bic:.2f}" for m in models]}

extra_df = pd.DataFrame(extra_rows, index=names).T
extra_df.index.name = df_clean.index.name

df_final = pd.concat([df_clean, extra_df])

display(HTML(df_final.to_html()))

In [ ]:
from statsmodels.stats.anova import anova_lm

# Joint F-test for DOW dummies
dow_dummies = ['C(day_name)[T.Monday]', 'C(day_name)[T.Saturday]',
               'C(day_name)[T.Sunday]', 'C(day_name)[T.Thursday]',
               'C(day_name)[T.Tuesday]', 'C(day_name)[T.Wednesday]']

hypotheses = ', '.join([f'({d} = 0)' for d in dow_dummies])
f_test = model_base2.f_test(hypotheses)

print(f"Joint F-test for DOW dummies:")
print(f"F-statistic : {f_test.fvalue:.4f}")
print(f"p-value     : {f_test.pvalue:.4f}")

**Comments**:

- The AR terms and Sunday effect are stable in magnitude and significance across all specifications, confirming they are not sensitive to media variable inclusion.

- Day-of-week dummies are retained as a group despite only Sunday reaching individual significance — the joint F-test confirms meaningful calendar variation that would otherwise confound media coefficients.

- `DUMMY_SEARCH` fades progressively as media variables enter, turning negative and insignificant once `Search_sat` is included.

  This confirms it was capturing the search regime shift rather than any independent structural break, and justifies its exclusion from the final spec.

- `CPD_sat` is insignificant across all specifications it appears in, consistent with the erratic spend-to-click behaviour observed in EDA. It is excluded from the final model.

- `WEB_MENTIONS` adds nothing beyond base2 and is dropped from all subsequent specs.

- When Search is split into always-on and broad components, SearchBroad enters with a negative, non-significant coefficient while SearchAO retains significance.

  This indicates that in OLS, the combined Search_sat specification is preferred as broad search does not contribute independently once always-on is controlled for.
  
  The split is reserved for the Meridian framework where Bayesian estimation handles correlated channels more robustly.

### Final baseline model

The following specification is carried forward as the baseline for grid search.

In [ ]:
formula_all = '''log_Trial ~ log_Trial_lag1 + log_Trial_lag2
              + C(day_name)
              + TV_RAI_adstock1 + TV_RAI_adstock2 + TV_RAI_adstock3 + TV_RAI_adstock4
              + TV_Paid_adstock + Radio_adstock
              + Search_sat
              + CPC_sat'''

model_all = smf.ols(formula_all, data=data_l2).fit(cov_type='HAC', cov_kwds={'maxlags': 7})

In [ ]:
model_all.summary()

## 4.2. Grid search for alpha & beta

In [ ]:
alpha_grid    = [0.1, 0.2, 0.3, 0.5, 0.7]
beta_pct_grid = [0.05, 0.15, 0.25, 0.35, 0.40]  # fraction of max adstock

In [ ]:
# ── Base columns that never change ────────────────────────────────────────────
# (TV and Radio adstocks are fixed — compute once outside the loop)
df = data_l2.copy()  # your dataframe with lags already created

In [ ]:
# ── Grid search ───────────────────────────────────────────────────────────────
records = []

for a_search, a_cpc in itertools.product(alpha_grid, alpha_grid):

    # Adstock with current alphas
    search_ads = geometric_adstock(df['SEARCH_spend'], a_search)
    cpc_ads    = geometric_adstock(df['CPC/CPM_spend'], a_cpc)

    for bp_search, bp_cpc in itertools.product(beta_pct_grid, beta_pct_grid):

        # Beta = fraction of max adstock
        beta_search = max(search_ads.max() * bp_search, 1e-3)
        beta_cpc    = max(cpc_ads.max()    * bp_cpc,    1e-3)

        # Saturation
        df['Search_sat'] = atan_saturation(search_ads, beta_search)
        df['CPC_sat']    = atan_saturation(cpc_ads,    beta_cpc)

        tmp = df.dropna().copy()

        formula = '''log_Trial ~ log_Trial_lag1 + log_Trial_lag2
                   + C(day_name)
                   + TV_RAI_adstock1 + TV_RAI_adstock2 + TV_RAI_adstock3 + TV_RAI_adstock4
                   + TV_Paid_adstock + Radio_adstock
                   + Search_sat + CPC_sat'''

        m = smf.ols(formula, data=tmp).fit(
            cov_type='HAC', cov_kwds={'maxlags': 7}
        )

        # Only keep if ALL media coefficients are positive (business constraint)
        media_vars = ['TV_RAI_adstock1','TV_RAI_adstock2','TV_RAI_adstock3',
                      'TV_RAI_adstock4','TV_Paid_adstock','Radio_adstock',
                      'Search_sat','CPC_sat']

        if all(m.params[v] > 0 for v in media_vars):
            records.append({
                'alpha_Search'  : a_search,
                'alpha_CPC'     : a_cpc,
                'beta_pct_Search': bp_search,
                'beta_pct_CPC'  : bp_cpc,
                'beta_Search'   : round(beta_search, 2),
                'beta_CPC'      : round(beta_cpc, 2),
                'adj_r2'        : m.rsquared_adj,
                'aic'           : m.aic,
            })

In [ ]:
# ── Results
results_df = pd.DataFrame(records).sort_values('aic')
print(f"Total combinations tested : {len(alpha_grid)**2 * len(beta_pct_grid)**2}")
print(f"Combinations passing check: {len(records)}")
print("\nTop 10 configurations:")
print(results_df.head(10).to_string(index=False))

Decay ($\alpha$) and saturation ($\beta$) parameters for Search and CPC were selected via grid search rather than fixed a priori.

Both parameters interact - a higher decay rate produces a larger adstock series, which shifts the effective saturation threshold - so they need to be optimised jointly.

The grids were kept intentionally sparse.

- For $\alpha$, values were drawn from {0.1, 0.2, 0.3, 0.5, 0.7}, covering the plausible range from fast decay (digital) to slow decay (broadcast), without testing every intermediate value.
- For $\beta$, five fractions of the channel's maximum adstock were tested, anchoring the saturation threshold relative to each channel's own scale rather than using absolute values.

Each combination was estimated on the full model and kept only if all media coefficients were positive — a hard business constraint. All passed, so AIC was used as the sole selection criterion.

The top configurations cluster tightly, indicating the objective surface is flat around the optimum.

The selected configuration achieves the best AIC and uses $\alpha_{Search} = 0.1$, $\alpha_{CPC} = 0.1$, $\beta_{Search} \approx 668$, and $\beta_{CPC} \approx 11222$. This specification is used in the final model reported below.

These values are consistent with prior expectations: low decay rates reflect the short-lived carryover typical of digital channels, while the estimated saturation parameters indicate diminishing returns at higher levels of media pressure.

In [ ]:
# Rebuild final model with best grid search parameters
best = results_df.iloc[0]

search_ads = geometric_adstock(data_l2['SEARCH_spend'],   best['alpha_Search'])
cpc_ads    = geometric_adstock(data_l2['CPC/CPM_spend'],  best['alpha_CPC'])

data_l2['Search_sat'] = atan_saturation(search_ads, best['beta_Search'])
data_l2['CPC_sat']    = atan_saturation(cpc_ads,    best['beta_CPC'])

formula_final = '''log_Trial ~ log_Trial_lag1 + log_Trial_lag2
                 + C(day_name)
                 + TV_RAI_adstock1 + TV_RAI_adstock2 + TV_RAI_adstock3 + TV_RAI_adstock4
                 + TV_Paid_adstock + Radio_adstock
                 + Search_sat + CPC_sat'''

model_final = smf.ols(formula_final, data=data_l2).fit(cov_type='HAC', cov_kwds={'maxlags': 7})
print(model_final.summary())

## 4.3. Interactions

The awareness-to-intent pathway motivates testing TV × digital interactions specifically: broadcast exposure raises brand awareness, which may amplify the effectiveness of downstream digital channels.

Radio × Search was also tested given the same theoretical logic.

CPC interactions were included as a weaker-theory alternative.

No other combinations were tested — channel pairs without a plausible sequential mechanism were excluded a priori.

In [ ]:
tmp = data_l2.copy()

# Aggregate TV measures
tmp['rai_tv_adstock'] = tmp[['TV_RAI_adstock1','TV_RAI_adstock2',
                              'TV_RAI_adstock3','TV_RAI_adstock4']].sum(axis=1)
tmp['tot_tv_adstock'] = tmp['rai_tv_adstock'] + tmp['TV_Paid_adstock']

# Interaction terms
tmp['TVtot_x_Search']  = tmp['tot_tv_adstock']  * tmp['Search_sat']
tmp['TVpaid_x_Search'] = tmp['TV_Paid_adstock']  * tmp['Search_sat']
tmp['TVrai_x_Search']  = tmp['rai_tv_adstock']   * tmp['Search_sat']
tmp['TVtot_x_CPC']     = tmp['tot_tv_adstock']   * tmp['CPC_sat']
tmp['Radio_x_Search']  = tmp['Radio_adstock']    * tmp['Search_sat']

# Two base formulas depending on RAI structure needed
base_individual = '''log_Trial_lag1 + log_Trial_lag2 + C(day_name)
                   + TV_RAI_adstock1 + TV_RAI_adstock2 + TV_RAI_adstock3 + TV_RAI_adstock4
                   + TV_Paid_adstock + Radio_adstock + Search_sat + CPC_sat'''

base_aggregated = '''log_Trial_lag1 + log_Trial_lag2 + C(day_name)
                   + rai_tv_adstock
                   + TV_Paid_adstock + Radio_adstock + Search_sat + CPC_sat'''

# Specs: (interaction terms, which base to use)
specs = {
    'No interaction':                     ([], base_individual),
    'Total TV x Search':                  (['TVtot_x_Search'], base_aggregated),
    'Paid TV x Search':                   (['TVpaid_x_Search'], base_individual),
    'RAI TV x Search':                    (['TVrai_x_Search'], base_aggregated),  # uses aggregated base
    'Total TV x CPC':                     (['TVtot_x_CPC'], base_aggregated),
    'Paid TV x Search + RAI TV x Search': (['TVpaid_x_Search', 'TVrai_x_Search'], base_aggregated),
    'Radio x Search':                     (['Radio_x_Search'], base_individual),
}

fitted = {
    name: smf.ols(
        f'log_Trial ~ {base}' + ((' + ' + ' + '.join(terms)) if terms else ''),
        data=tmp
    ).fit(cov_type='HAC', cov_kwds={'maxlags': 7})
    for name, (terms, base) in specs.items()
}

# Comparison table
interaction_vars = ['TVtot_x_Search', 'TVpaid_x_Search', 'TVrai_x_Search',
                    'TVtot_x_CPC', 'Radio_x_Search']

rows = []
for name, m in fitted.items():
    inter_coef = {v: f"{m.params[v]:.4f} (p={m.pvalues[v]:.3f})"
                  for v in interaction_vars if v in m.params.index}
    rows.append({
        'Model':       name,
        'Adj R²':      round(m.rsquared_adj, 4),
        'AIC':         round(m.aic, 4),
        'DW':          round(durbin_watson(m.resid), 4),
        'Interaction': ', '.join(inter_coef.keys()),
        'Coef (p)':    ', '.join(inter_coef.values()),
    })

display(pd.DataFrame(rows).sort_values('AIC'))

**RAI TV × Search** is the only significant interaction, delivering the best AIC
across all specifications.

The positive coefficient confirms that RAI TV amplifies search effectiveness, consistent with the awareness-to-intent story. Paid TV shows no equivalent synergy: when entered alongside RAI TV × Search, its interaction term is negative and insignificant, suggesting it operates as a direct-response channel rather than an awareness driver.

All remaining interactions  are insignificant and do not improve on the baseline.

In [ ]:
model_interaction = fitted['RAI TV x Search']
print(model_interaction.summary())

## 4.4. Models

1. We include 2 lags as regressors to absorb the strong temporal autocorrelation in daily trials (confirmed by PACF and Durbin-Watson diagnostics). This is a deliberate modelling choice: without AR lags, residuals show severe autocorrelation (DW ≈ 1.0), invalidating inference.

    The tradeoff is that AR terms partially absorb media-correlated persistence, producing conservative media elasticities. A robustness check without AR lags confirms coefficients inflate substantially, consistent with this known MMM tradeoff.  
    
    We treat AR-model elasticities as defensible lower bounds.

In [ ]:
# ── Robustness check: OLS without AR lags ─────────────────────────────────────
formula_no_ar = '''log_Trial ~ C(day_name)
                 + TV_RAI_adstock1 + TV_RAI_adstock2 + TV_RAI_adstock3 + TV_RAI_adstock4
                 + TV_Paid_adstock + Radio_adstock
                 + Search_sat + CPC_sat'''

model_no_ar = smf.ols(formula_no_ar, data=data_l2).fit(cov_type='HAC', cov_kwds={'maxlags': 7})

# ── Coefficient comparison ─────────────────────────────────────────────────────
media_vars = ['Search_sat', 'CPC_sat', 'TV_Paid_adstock', 'Radio_adstock',
              'TV_RAI_adstock1', 'TV_RAI_adstock2', 'TV_RAI_adstock3', 'TV_RAI_adstock4']

comparison = pd.DataFrame({
    'With AR(2)': model_final.params[media_vars],
    'Without AR(2)': model_no_ar.params[media_vars],
})
comparison['Δ%'] = ((comparison['Without AR(2)'] - comparison['With AR(2)'])
                    / comparison['With AR(2)'].abs() * 100).round(1)
print(comparison.round(4))

# ── Model-level stats ──────────────────────────────────────────────────────────
from statsmodels.stats.stattools import durbin_watson

compare = pd.DataFrame({
    'With AR(2)':    [model_final.rsquared, model_final.rsquared_adj,
                      model_final.aic, model_final.bic,
                      durbin_watson(model_final.resid)],
    'Without AR(2)': [model_no_ar.rsquared, model_no_ar.rsquared_adj,
                      model_no_ar.aic, model_no_ar.bic,
                      durbin_watson(model_no_ar.resid)],
}, index=['R²', 'Adj. R²', 'AIC', 'BIC', 'Durbin-Watson'])
print(compare.round(4))

2. Use of two models

   Interacting RAI TV with `Search_sat` requires a structural choice: four individual flights or one aggregated variable? Individual interactions are not viable - each flight is active on only ~0.4% of days, too sparse for stable estimates.
   
   This leads to two complementary specifications:
    - **Creative flights model** — four individual RAI flights, no interaction.
    
    Used for elasticities and creative-level analysis.
    - **Interaction model** — single aggregated `rai_tv_adstock` × `Search_sat`.
    
    Used to test cross-channel synergy.

    The two models answer different questions and are not in conflict.

In [ ]:
model_1 = smf.ols(f'log_Trial ~ {base_individual}', data=tmp).fit(cov_type='HAC', cov_kwds={'maxlags': 7})
model_2 = smf.ols(f'log_Trial ~ {base_aggregated} + TVrai_x_Search', data=tmp).fit(cov_type='HAC', cov_kwds={'maxlags': 7})

comparison = summary_col(
    [model_1, model_2],
    stars=True,
    float_format='%.3f',
    model_names=['Model 1\n(individual flights)', 'Model 2\n(RAI x Search)'],
    info_dict={
        'R-squared Adj.': lambda x: f"{x.rsquared_adj:.3f}",
        'AIC':            lambda x: f"{x.aic:.2f}",
        'BIC':            lambda x: f"{x.bic:.2f}",
        'N':              lambda x: f"{int(x.nobs)}"
    }
)

# Clean up — drop the std error rows
df = comparison.tables[0]
df_clean = df[df.index.str.strip() != '']

display(HTML(df_clean.to_html()))

The key observation is that all non-RAI coefficients remain stable across the two models, confirming the interaction finding is not an artefact of the restructuring.

| | Creative Flights Model | Interaction Model |
|---|---|---|
| RAI TV structure | 4 individual flights | single `rai_tv_adstock` |
| RAI × Search | — | ✓ |
| Purpose | creative differences, elasticity | cross-channel synergy |

**What stays stable:**
  - AR terms (`log_Trial_lag1`, `log_Trial_lag2`), Day-of-week effects,   `TV_Paid_adstock` and `Radio_adstock`: essentially unchanged

- `Search_sat` and `CPC_sat`: slight drop in the interaction model, expected because part of search's effect is now captured through `TVrai_x_Search`

**What changes :**

- RAI TV: disaggregated positive flights (creative flights model) become a   single negative main effect in the interaction model. This is expected:  the standalone `rai_tv_adstock` coefficient represents RAI's effect when
  `Search_sat` = 0, which never occurs in the data.
  
  The meaningful quantity is always the **net effect** at realistic search levels (see elasticity section).

- The interaction term `TVrai_x_Search` is strongly significant (p < 0.001), providing direct evidence that RAI TV amplifies search effectiveness beyond what either channel achieves independently

## 4.5. Model diagnostics

In [ ]:
# Get the design matrix from model_final (excluding the constant)
X = model_final.model.exog
cols = model_final.model.exog_names

vif_df = pd.DataFrame({
    'Variable': cols,
    'VIF': [variance_inflation_factor(X, i) for i in range(X.shape[1])]
}).sort_values('VIF', ascending=False)

print(vif_df.to_string(index=False))

In [ ]:
X1, cols1 = model_final.model.exog, model_final.model.exog_names
X2, cols2 = model_interaction.model.exog, model_interaction.model.exog_names

vif1 = pd.DataFrame({'Variable': cols1, 'VIF_final': [variance_inflation_factor(X1, i) for i in range(X1.shape[1])]})
vif2 = pd.DataFrame({'Variable': cols2, 'VIF_interaction': [variance_inflation_factor(X2, i) for i in range(X2.shape[1])]})

vif_df = vif1.merge(vif2, on='Variable', how='outer').sort_values('VIF_final', ascending=False)

print(vif_df.to_string(index=False))

VIF was computed for all variables in `model_final`. Excluding the intercept (inflated by construction), all variables fall well below the threshold of concern:

No variable exceeds 3.5, indicating no meaningful ulticollinearity. Ridge regression is not warranted; OLS estimates are stable and interpretable.

**Note on `model_interaction`:** The high VIF for `rai_tv_adstock` and
`TVrai_x_Search` is expected — interaction terms are by construction
correlated with their constituent variables. This does not affect the
interaction coefficient, its significance, or the model's overall fit.

In [ ]:
def plot_diagnostics(model, df, title):
    resid       = model.resid
    fitted      = model.fittedvalues
    actual_lvl  = np.exp(df['log_Trial'])
    fitted_lvl  = np.exp(fitted)

    fig = plt.figure(figsize=(16, 10))
    gs  = gridspec.GridSpec(3, 2, hspace=0.35, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(df.index, resid, linewidth=0.9, color='steelblue')
    ax1.axhline(0, color='red', linestyle='--', linewidth=0.8)
    ax1.set_title('Residuals over Time')
    ax1.set_xlabel('Date'); ax1.set_ylabel('Residual')

    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(df.index, df['log_Trial'], label='Actual',  linewidth=0.9)
    ax2.plot(df.index, fitted,          label='Fitted',  linewidth=0.9, linestyle='--', color = 'goldenrod')
    ax2.set_title('Actual vs Fitted (log scale)')
    ax2.set_xlabel('Date'); ax2.legend()

    ax3 = fig.add_subplot(gs[1, 1])
    ax3.plot(df.index, actual_lvl, label='Actual',  linewidth=0.9)
    ax3.plot(df.index, fitted_lvl, label='Fitted',  linewidth=0.9, linestyle='--', color = 'goldenrod')
    ax3.set_title('Actual vs Fitted (Trial levels)')
    ax3.set_xlabel('Date'); ax3.legend()

    ax4 = fig.add_subplot(gs[2, 0])
    plot_acf(resid, lags=20, ax=ax4, zero=False)
    ax4.set_title('ACF of Residuals')

    ax5 = fig.add_subplot(gs[2, 1])
    stats.probplot(resid, plot=ax5)
    ax5.set_title('Q-Q Plot of Residuals')

    plt.suptitle(f'Model Diagnostics ({title})', fontweight='bold')
    plt.show()

In [ ]:
plot_diagnostics(model_final, tmp, 'Creative Flights')

In [ ]:
plot_diagnostics(model_interaction, tmp, 'Interaction')

For both models:

*Residuals over time*
- Residuals scatter randomly around zero throughout the sample with no visible trend or systematic pattern, indicating no major misspecification.
- Variance is slightly elevated in the early period, consistent with the higher volatility observed in the raw trial series before the search campaign launch.

*Actual vs fitted*
- The model tracks the observed series closely in both log scale and trial levels, including through the November–December TV burst.
- The early-May spike is not fully captured, likely reflecting the erratic CPC/CPM activity in that period; the residual is isolated rather than systematic and does not indicate structural misfit.

*ACF of residuals*
- All lags fall within the confidence bands, confirming no residual autocorrelation.
- The AR(2) specification successfully absorbs the temporal dependence in daily trials.

*Q-Q plot*
- Rsiduals follow the normal reference line closely through the bulk of the distribution.
- Slight tail deviations are expected with daily data subject to occasional spikes and are not a concern for inference given the use of HAC standard errors throughout. The upper tail deviation is marginally more pronounced in the interaction model but remains non-material.

In [ ]:
def model_summary_row(model, label):
    resid       = model.resid
    fitted      = model.fittedvalues
    actual_lvl  = np.exp(tmp['log_Trial'])
    fitted_lvl  = np.exp(fitted)
    ss_res      = np.sum((actual_lvl - fitted_lvl) ** 2)
    ss_tot      = np.sum((actual_lvl - actual_lvl.mean()) ** 2)
    return {
        'Model'              : label,
        'Adj R² (log)'       : round(model.rsquared_adj, 3),
        'R² (levels)'        : round(1 - ss_res / ss_tot, 3),
        'AIC'                : round(model.aic, 2),
        'BIC'                : round(model.bic, 2),
        'Durbin-Watson'      : round(durbin_watson(resid), 3),
        'α Search'           : best['alpha_Search'],
        'α CPC'              : best['alpha_CPC'],
        'β Search'           : round(best['beta_Search'], 2),
        'β CPC'              : round(best['beta_CPC'], 2),
    }

summary_df = pd.DataFrame([
    model_summary_row(model_final,       'Creative Flights'),
    model_summary_row(model_interaction, 'Interaction'),
]).set_index('Model')

summary_df

# 5. Second screen effect

A key cross-channel question in this project is whether TV exposure triggers immediate digital responses.

This is the classic **second-screen effect**: consumers see a TV ad and, at the same time or shortly
after, they use another device (typically a smartphone) to search for the brand, visit the website,
or move closer to conversion.

In this dataset, the most natural proxies for second-screen behaviour are:
- **CLIC_SEARCH**: total search clicks
- **clic_search_alwayson**: always-on search clicks
- **SEARCH_spend / search_alwayson_spend**: search investment
- **WEB_MENTIONS**: online buzz / conversation

> **Note on broad search**:
> - The dataset contains two search sub-channels: clic_search_alwayson (active throughout May–December 2016) and clic_search_broad (zero before October 2016).
> - Because broad search is structurally absent for the first five months of the campaign, it cannot be used in any dynamic model estimated on the full sample. There is no meaningful variation to identify its effect.

Our working hypotheses are:

1. TV should increase search activity, especially on the same day or with a very short lag.
2. The effect may differ across TV types:
    - Paid TV (`TV_Grp_Mediaset_Premium_Sky`)
    - Public TV / RAI creatives (`TV_Grp_Rai3_OnAir1` … `OnAir4`)
    
3. If second-screen behaviour is real, TV may also create a **TV × Search synergy** in the Trial model.

The EDA already showed a visible positive association between TV GRPs and search clicks.

In this section we move from that descriptive observation to more structured evidence:
- first with conditional comparisons and lead–lag timing,
- then with dynamic regression models.

## 5.1 Variable setup

Minimal setup: we only need log-transformed search outcomes and their lags.

Everything else (adstock, saturation, day-of-week, Trial lags) is created from the feature-engineering and modelling sections.

In [ ]:
df_ss = data_l2.copy()

# Log transforms for search outcomes (np.log to match the main notebook).
# WEB_MENTIONS uses log1p because it contains genuine zeros.
for col in ['CLIC_SEARCH', 'clic_search_alwayson', 'SEARCH_spend',
            'search_alwayson_spend']:
    df_ss[f'log_{col}'] = np.log(df_ss[col])

df_ss['log_WEB_MENTIONS'] = np.log1p(df_ss['WEB_MENTIONS'])

# Lagged dependent variables for the dynamic search-response models
df_ss['log_CLIC_SEARCH_lag1']          = df_ss['log_CLIC_SEARCH'].shift(1)
df_ss['log_clic_search_alwayson_lag1'] = df_ss['log_clic_search_alwayson'].shift(1)
df_ss['log_WEB_MENTIONS_lag1']         = df_ss['log_WEB_MENTIONS'].shift(1)

df_ss_model = df_ss.dropna().copy()
print("Second-screen working sample:", df_ss_model.shape)

## 5.2 Descriptive evidence: TV-on vs TV-off

Before running regressions, we check whether search activity is visibly higher on days with TV pressure.

This is only **descriptive, not causal** — TV may be scheduled together with other media pushes, and both search and trials have strong time dependence.

We focus on two views that go beyond what the EDA already showed:

1. A **conditional comparison** (TV-on vs TV-off mean/median lift) — this quantifies the raw gap in a way the scatter plot alone does not.

2. An **intensity gradient** (quartiles of TV pressure among TV-on days) — this tests whether the relationship is monotonic, not just binary.

In [ ]:
df_ss_model['TV_on'] = (df_ss_model['tot_tv'] > 0).astype(int)

# --- TV-on vs TV-off lift table ---
off = df_ss_model[df_ss_model['TV_on'] == 0]
on  = df_ss_model[df_ss_model['TV_on'] == 1]

lift_table = pd.DataFrame({
    'Metric':      ['CLIC_SEARCH', 'clic_search_alwayson', 'WEB_MENTIONS', 'Trial'],
    'TV_off_mean': [off['CLIC_SEARCH'].mean(), off['clic_search_alwayson'].mean(),
                    off['WEB_MENTIONS'].mean(), off['Trial'].mean()],
    'TV_on_mean':  [on['CLIC_SEARCH'].mean(),  on['clic_search_alwayson'].mean(),
                    on['WEB_MENTIONS'].mean(),  on['Trial'].mean()]
})
lift_table['Lift_%'] = 100 * (lift_table['TV_on_mean'] / lift_table['TV_off_mean'] - 1)

print("Lift on TV-on days vs TV-off days:")
display(lift_table.round(2))

Search clicks roughly **double** on TV-on days (+99% for CLIC_SEARCH, +84% for always-on).

Trials also rise substantially (+48%).

In [ ]:
# --- Intensity gradient: search across quartiles of positive TV days ---
tv_pos = df_ss_model[df_ss_model['tot_tv'] > 0].copy()
tv_pos['tv_bin'] = pd.qcut(tv_pos['tot_tv'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
quartile_summary = tv_pos.groupby('tv_bin', observed=False)[['CLIC_SEARCH', 'clic_search_alwayson', 'Trial']].mean()

print("\nSearch activity by TV intensity quartile (TV-on days only):")
display(quartile_summary.round(2))

The quartile analysis shows a **monotonic** increase: the higher the TV pressure, the more search activity, consistent with a dose–response pattern.

These patterns are suggestive but could reflect confounding (e.g., media scheduling correlation).

The next two subsections add timing and regression controls.

## 5.3 Lead–lag evidence

Second-screen behaviour should be **fast**.

If TV triggers search, the correlation should be strongest:
- on the **same day**,
- or with a **very short lag**.

We compute lead–lag correlations between TV variables and digital-response variables.

The goal is not to claim causality from correlation, but to check whether the **timing is consistent** with a second-screen mechanism.

In [ ]:
tv_vars = ['tot_tv',
    'TV_Grp_Mediaset_Premium_Sky',
    'TV_Grp_Rai',
    'TV_Grp_Rai3_OnAir1',
    'TV_Grp_Rai3_OnAir2',
    'TV_Grp_Rai3_OnAir3',
    'TV_Grp_Rai3_OnAir4']

digital_targets = ['CLIC_SEARCH',
    'clic_search_alwayson',
    'SEARCH_spend',
    'search_alwayson_spend',
    'WEB_MENTIONS']

lead_lag_rows = []
for tv in tv_vars:
    for y in digital_targets:
        for k in range(4):
            corr = df_ss_model[tv].corr(df_ss_model[y].shift(-k))
            lead_lag_rows.append({
                'TV_variable':     tv,
                'Digital_outcome': y,
                'lag_k':           k,
                'corr':            corr
            })

lead_lag_df = pd.DataFrame(lead_lag_rows)

# Show top correlations per digital outcome
# for y in digital_targets:
#     print(f"\nTop correlations for {y}")
#     display(lead_lag_df[lead_lag_df['Digital_outcome'] == y]
#         .sort_values('corr', ascending=False)
#         .head(10)
#         .round(3))

**Heatmap: total TV vs digital outcomes at different lags**

In [ ]:
pivot_tot_tv = (
    lead_lag_df[lead_lag_df['TV_variable'] == 'tot_tv']
    .pivot(index='Digital_outcome', columns='lag_k', values='corr'))
plt.figure(figsize=(8, 4))
sns.heatmap(pivot_tot_tv, annot=True, fmt=".2f", cmap='Reds')
plt.title('Lead–lag correlations: total TV vs digital-response variables')
plt.xlabel('Lag k (days ahead)')
plt.ylabel('')
plt.show()

The heatmap confirms two key patterns:
- **Search clicks** show the strongest correlation with TV at **lag 0** (same day), exactly what a second-screen mechanism predicts.
- `WEB_MENTIONS` show near-zero correlations at all lags - TV does not appear to drive organic buzz, only active search behaviour.
- Among TV types, **Paid TV** (Mediaset/Premium/Sky) shows persistent correlations that increase slightly at lags 1–3, while **RAI** shows a sharper same-day peak.

## 5.4 Econometric evidence: does TV increase search?

The descriptive patterns are suggestive, but we need a more disciplined test.

We estimate dynamic search-response models where the dependent variable is:
- `log(CLIC_SEARCH)` for total search clicks
- `log(clic_search_alwayson)` for always-on search clicks
- `log1p(WEB_MENTIONS)` for online buzz

Each model controls for:
- persistence in the dependent variable (lag 1),
- day-of-week seasonality,
- TV pressure split into paid TV and RAI creative flights.

We use **HAC / Newey–West standard errors** because daily media data often show heteroskedasticity and autocorrelation.

In [ ]:
# --- Model A: total search clicks ---
formula_ss_search = '''
    log_CLIC_SEARCH ~ log_CLIC_SEARCH_lag1 + C(day_name)
        + TV_Paid_adstock
        + TV_RAI_adstock1 + TV_RAI_adstock2 + TV_RAI_adstock3 + TV_RAI_adstock4
'''
model_ss_search = smf.ols(formula_ss_search, data=df_ss_model).fit(
    cov_type='HAC', cov_kwds={'maxlags': 7}
)

In [ ]:
# --- Model B: always-on search clicks ---
formula_ss_alwayson = '''
    log_clic_search_alwayson ~ log_clic_search_alwayson_lag1 + C(day_name)
        + TV_Paid_adstock
        + TV_RAI_adstock1 + TV_RAI_adstock2 + TV_RAI_adstock3 + TV_RAI_adstock4
'''
model_ss_alwayson = smf.ols(formula_ss_alwayson, data=df_ss_model).fit(
    cov_type='HAC', cov_kwds={'maxlags': 7})

In [ ]:
# --- Model C: web mentions ---
formula_ss_mentions = '''
    log_WEB_MENTIONS ~ log_WEB_MENTIONS_lag1 + C(day_name)
        + TV_Paid_adstock
        + TV_RAI_adstock1 + TV_RAI_adstock2 + TV_RAI_adstock3 + TV_RAI_adstock4
'''
model_ss_mentions = smf.ols(formula_ss_mentions, data=df_ss_model).fit(
    cov_type='HAC', cov_kwds={'maxlags': 7})

In [ ]:
summary = summary_col(
    [model_ss_search, model_ss_alwayson, model_ss_mentions],
    stars=True,
    float_format='%.3f',
    model_names=['A: CLIC_SEARCH', 'B: alwayson', 'C: WEB_MENTIONS']
)

df_sum = summary.tables[0]
df_sum = df_sum[df_sum.index.str.strip() != '']

# Add fit statistics
models = [model_ss_search, model_ss_alwayson, model_ss_mentions]
names  = ['A: CLIC_SEARCH', 'B: alwayson', 'C: WEB_MENTIONS']

extra = pd.DataFrame({
    'AIC': [f"{m.aic:.2f}" for m in models],
    'BIC': [f"{m.bic:.2f}" for m in models],
    'DW':  [f"{durbin_watson(m.resid):.3f}" for m in models],
}, index=names).T
extra.index.name = df_sum.index.name
df_sum = pd.concat([df_sum, extra])

display(HTML(df_sum.to_html()))

**Interpretation of the search-response models**

The evidence is strongest for the pathway: **TV → Search**

In particular:
- **Paid TV** shows a clear and statistically significant positive relationship with search activity in both Model A and Model B.
- Some **RAI creative flights** also appear to boost always-on search, although the effect is less uniform across flights.
- By contrast, the link between TV and **web mentions** is much weaker and less stable.

This pattern is exactly what we would expect from a second-screen mechanism:
> TV seems to stimulate **active information-seeking behaviour** (search) more than passive online buzz.


## 5.5 Connecting to the Trial model

The search-response models above establish the first link in the second-screen chain: **TV → Search**.

The second link — whether this TV-driven search activity translates
into additional Trial uplift — was already tested in Section 4.3 (Interactions).

That analysis found that **RAI TV × Search** is the only significant interaction, while Paid TV × Search is not.

Combining both pieces, the full second-screen mechanism is:

| Step | Finding | Where |
|------|---------|-------|
| TV → Search | Paid TV significantly boosts search clicks (same-day) | Section 5.4 |
| RAI TV × Search → Trial | RAI TV amplifies search conversion | Section 4.3 |
| Paid TV × Search → Trial | Not significant — Paid TV drives trials independently | Section 4.3 |


## 5.6 Business takeaway

From a managerial perspective, the second-screen analysis suggests three main conclusions:

1. **TV is not only an upper-funnel awareness channel.**

    It also appears to stimulate immediate lower-funnel behaviour, especially search.

2. **Search should not be planned in isolation from TV.**

    When TV pressure is on air, users seem more likely to react digitally, so search coverage is strategically important during TV flights.

3. **The channel type matters for the mechanism.**
    - **RAI TV** works primarily through an awareness-to-intent pathway: it raises brand salience, amplifying search effectiveness (significant interaction).

    - **Paid TV** (Mediaset/Sky) works as a direct-response driver with its own independent Trial effect, without an additional search synergy.

Overall, the second-screen effect in this dataset is best interpreted as:

> TV helps create demand that quickly materialises in search activity, and search then helps convert that demand into Trials.
>
> RAI TV in particular amplifies search effectiveness through an awareness-to-intent mechanism, while Paid TV drives trials independently.

# 6. Two-stage model

**Motivation**

Sections 4 and 5 reveal that TV and search are linked through two distinct mechanisms.
- TV_Paid drives search volume — consumers see an ad and immediately search for the brand.
- RAI does not drive search volume; instead it amplifies search effectiveness, making each search more likely to convert into a trial.

This matters for measurement.

Because TV_Paid inflates search clicks, the benchmark's spend-based search coefficient absorbs some of that TV-driven demand — it is not purely capturing organic search investment. Simply swapping
spend for raw clicks would reintroduce this TV component alongside TV itself, creating double-counting.

The two-stage solves this by replacing `Search_sat` with **Stage-1 fitted clicks** — the component of search activity predicted by TV pressure, seasonality, and momentum.
- Stage 1 is `model_ss_search` from Section 5.4;
- Stage 2 reruns the trial model with this cleaner search measure. The goal is to quantify how much of the benchmark's search coefficient reflects the TV_Paid→Search→Trials pathway rather than direct search investment.

In [ ]:
# Section 6 helpers

def fit_hac_ols(formula, df, maxlags=7):
    """Fit OLS with HAC (Newey-West) standard errors."""
    return smf.ols(formula=formula, data=df).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": maxlags})

def model_stats(model):
    """Return a dict of key fit statistics for quick comparison."""
    return {
        "nobs":          int(model.nobs),
        "adj_r2":        model.rsquared_adj,
        "aic":           model.aic,
        "bic":           model.bic,
        "durbin_watson": durbin_watson(model.resid)}

def plot_actual_vs_fitted(ax, x, actual, fitted, title):
    """Overlay actual vs fitted on a given axes."""
    ax.plot(x, actual, label="Actual")
    ax.plot(x, fitted, label="Fitted", color = 'goldenrod')
    ax.set_title(title)
    ax.legend()

## 6.1 Stage 1: Fitted Search Variable

We take fitted values from `model_ss_search` (Section 5.4), back-transform to levels, and apply arctan saturation at the median of the fitted series — the same default used in Section 2.2.

In [ ]:
# Back-transform fitted log(CLIC_SEARCH) → levels, clip at zero
df_ss_model = df_ss_model.copy()
df_ss_model["CLIC_SEARCH_fitted"] = np.expm1(model_ss_search.fittedvalues).clip(lower=0)

# Merge fitted values into the main dataframe
# (only defined for rows where Stage 1 was estimated; rest stay NaN)
data["CLIC_SEARCH_fitted"] = np.nan
data.loc[df_ss_model.index, "CLIC_SEARCH_fitted"] = df_ss_model["CLIC_SEARCH_fitted"]

# Saturation: beta = median of fitted clicks (same default as Section 2.2)
beta_search_fitted = max(float(data["CLIC_SEARCH_fitted"].dropna().median()), 1e-3)

data["Search_fitted_sat"] = atan_saturation(
    data["CLIC_SEARCH_fitted"].fillna(0),
    beta_search_fitted
)

print(f"beta_search_fitted = {round(beta_search_fitted, 3)}")
print(f"Search_fitted_sat — non-null: {data['Search_fitted_sat'].notna().sum()}, "
      f"range: [{data['Search_fitted_sat'].min():.4f}, {data['Search_fitted_sat'].max():.4f}]")

## 6.2 Benchmark

`model_all2` from Section 4.1, reprinted here for side-by-side comparison. Both models use median-based saturation parameters; the **only difference** between them is the search variable.

In [ ]:
model_benchmark = model_final

## 6.3 Stage 2

In [ ]:
data_l2["Search_fitted_sat"] = data.loc[data_l2.index, "Search_fitted_sat"].values

formula_2stage = '''
log_Trial ~ log_Trial_lag1 + log_Trial_lag2
            + C(day_name)
            + TV_RAI_adstock1 + TV_RAI_adstock2 + TV_RAI_adstock3 + TV_RAI_adstock4
            + TV_Paid_adstock + Radio_adstock
            + Search_fitted_sat + CPC_sat
'''

model_2stage = fit_hac_ols(formula_2stage, data_l2)

In [ ]:
comparison = summary_col(
    [model_benchmark, model_2stage],
    stars=True,
    float_format='%.3f',
    model_names=['Benchmark', '2stage', '2stage_interaction'],
    info_dict={
        'R-squared Adj.': lambda x: f"{x.rsquared_adj:.3f}",
        'AIC':            lambda x: f"{x.aic:.2f}",
        'BIC':            lambda x: f"{x.bic:.2f}"}
)

df = comparison.tables[0]
df_clean = df[df.index.str.strip() != ""]

display(HTML(df_clean.to_html()))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

plot_actual_vs_fitted(
    axes[0], data_l2.index, data_l2["Trial"],
    np.exp(model_2stage.fittedvalues), "Two-stage"
)

plot_acf(model_2stage.resid, ax=axes[1], lags=30, zero=False)
axes[1].set_title("Two-stage residuals")

plt.tight_layout()
plt.show()

## 6.4. Results


**Coefficient shifts:** Most coefficients move negligibly across specifications.
The notable exception is `TV_RAI_adstock1`, which loses significance in the
two-stage, suggesting its effect in the benchmark was partly proxying for
TV_Paid-driven search demand rather than a direct contribution to trials.

**Search coefficient:** The search coefficient drops when moving from
`Search_sat` to `Search_fitted_sat`, consistent with the Section 5 finding:
the benchmark's spend-based measure was absorbing some TV_Paid-driven
second-screen demand. The two-stage isolates the organic component.

**Fit:** The benchmark retains a slight edge in Adj. $R^2$ and AIC since spend captures total search investment while fitted clicks represent only the TV- and seasonality-explained component. Lower fit is the expected cost
of a cleaner measure..

**Bottom line:** The drop in the search coefficient quantifies the
`TV_Paid → Search → Trials` pathway the benchmark silently absorbs. Combined with
the RAI×Search interaction in Section 4, this completes the picture:
- TV_Paid fills the search funnel,
- RAI makes it more effective.

# 7. Findings

## 7.1. Model overview

Three complementary specifications are estimated, all sharing the same AR(2) backbone, day-of-week seasonality controls, and HAC standard errors (7 lags).

|Model | Key feature |
|---|---|
|`model_final`| **4 RAI creative flights entered individually**<br> Allows the model to recover differences in creative effectiveness across flights |
|`model_interaction`| **Aggregated RAI + RAI × Search interaction term** <br> Captures the cross-channel amplification mechanism: <br>*RAI TV raises brand awareness, which increases the effectiveness of search rather than simply adding to it additively*.|
|`model_2stage`| **Search instrumented on TV-predicted clicks** <br> Rather than treating search spend as the media input, it instruments for search using the portion of search activity predicted by spend, TV and seasonality. <br>This isolates the search demand that originates from consumer intent rather than media investment|

## 7.2 Impact of Media Variables on App Trials

In [ ]:
tv_radio_vars = {
    'TV_RAI_adstock1' : 'TV RAI – Flight 1',
    'TV_RAI_adstock2' : 'TV RAI – Flight 2',
    'TV_RAI_adstock3' : 'TV RAI – Flight 3',
    'TV_RAI_adstock4' : 'TV RAI – Flight 4',
    'TV_Paid_adstock' : 'TV Paid',
    'Radio_adstock'   : 'Radio',
}

digital_vars = {
    'CPC_sat'    : 'CPC/CPM',
    'Search_sat' : 'Search',
}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for ax, var_dict, title in [
    (ax1, tv_radio_vars, 'TV & Radio Channels'),
    (ax2, digital_vars,  'Digital Channels'),
]:
    labels = list(var_dict.values())
    x      = np.arange(len(labels))
    width  = 0.25

    for i, (model, name, color) in enumerate([
        (model_final,       'Creative Flights',       'lightskyblue'),
        (model_interaction, 'TV x Search Interaction Model', 'yellowgreen'),
        (model_2stage,      '2 - Stage Model',      'pink'),
    ]):
        coefs  = []
        errors = []
        for var in var_dict:
            if var == 'Search_sat' and model == model_2stage:
                v = 'Search_fitted_sat'
                coefs.append(model.params.get(v, np.nan))
                errors.append(1.96 * model.bse.get(v, np.nan))
            elif var in ['TV_RAI_adstock1','TV_RAI_adstock2',
                         'TV_RAI_adstock3','TV_RAI_adstock4'] and model == model_interaction:
                coefs.append(np.nan)
                errors.append(np.nan)
            else:
                coefs.append(model.params.get(var, np.nan))
                errors.append(1.96 * model.bse.get(var, np.nan))

        ax.bar(x + i * width, coefs, width,
               yerr=errors, capsize=3,
               label=name, color=color, alpha=0.8,
               edgecolor='white', error_kw={'elinewidth': 1.2, 'ecolor': 'black'})

    ax.set_xticks(x + width)
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Coefficient')
    ax.set_title(title)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.legend(fontsize=8)

fig.suptitle('Media Coefficients Across Model Specifications', fontweight='bold')
plt.tight_layout()
plt.show()

Looking at the final model results, several media variables show a statistically significant impact on app trials, summarized as following:

- **Search** has a strong positive effect (coef ≈ 0.349, p < 0.001). This means that increases in search activity are associated with higher app trials, which is consistent with the idea that search captures high-intent users.

- **TV RAI** creative flights all have positive and significant
coefficients (0.035–0.041), with Flight 3 performing best. Flights 2 and 4 perform at similar levels, while Flight 1,
though significant, carries more uncertainty, reflected in its wider confidence interval.

- **TV Paid** (Mediaset/Sky) is positive and significant (coef ≈ 0.021, p < 0.01), contributing a meaningful direct effect on top of its indirect role in driving search activity.

- **Radio** has a positive and significant coefficient (p ≈ 0.001). The
raw coefficient appears small relative to other channels, but this
reflects differences in input units rather than true ineffectiveness —
the elasticity comparison in section 7.6 shows Radio performing
comparably to TV Paid in the short run..

- Among digital display channels, **CPC_sat** is positive and highly significant (≈ 0.340, p < 0.001). This suggests that even after accounting for saturation effects, CPC campaigns still play a meaningful role in driving trials.

On the other hand, some variables were not included in the final model.
- **CPD** was dropped because of its unstable and noisy relationship with clicks and trials, making it difficult to estimate a reliable effect.
- Similarly, **Web Mentions** was excluded since it did not show a consistent or significant relationship with trials during earlier analysis.

Overall, the results indicate that Search is the strongest driver, while TV contributes both directly and through lagged effects, and other channels like Radio and CPC play supporting roles.

## 7.3 Cross-Channel Effects: Does TV Boost Search?

This section examines whether TV and search reinforce each other, and we find that TV affects trials both directly and indirectly through search.



### 7.3.1 — Does TV drive more search activity?

Yes. On Paid TV airing days, search clicks increase by about +99%, nearly doubling compared to non-airing days. This is also supported by the lead-lag analysis, where total TV has a strong same-day correlation with search clicks (≈ 0.63) and still meaningful effects at later lags (~0.44–0.46).
This confirms a strong immediate second-screen response.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

lift_pct = lift_table.set_index('Metric')['Lift_%']
labels   = ['Search Clicks', 'Always-on Search', 'Web Mentions', 'Trials']
colors   = ['mediumaquamarine' if v > 0 else 'tomato' for v in lift_pct.values]

bars = ax.bar(labels, lift_pct.values, color=colors, alpha=0.8, edgecolor='white')

# Annotate with the actual % values
for bar, val in zip(bars, lift_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1,
            f'{val:.0f}%', ha='center', fontsize=10, fontweight='bold')

ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('% lift on TV-on days vs TV-off days')
ax.set_title('Second-Screen Effect: Lift on TV Airing Days')
plt.tight_layout()
plt.show()

### 7.3.2 — Does TV make search more effective?

Yes, but the two TV types work differently.

- The second-screen models show that Paid TV drives search *volume* — on airing days, more people search.

- RAI TV, by contrast, amplifies search *conversion efficiency*:
`model_interaction` shows a positive and significant RAI × Search interaction term, meaning that on days with high RAI pressure, the same level of search activity generates more trials than it would otherwise.

In short, Paid TV brings users to the search bar; RAI TV makes them more likely to convert once they get there.

### 7.3.3 — How much of the search effect is TV-driven?

In the benchmark model, the search coefficient is 0.349, while in the two-stage model it drops to 0.292. This corresponds to roughly a 16% decrease, meaning that part of the original search effect was actually driven by TV.
This shows that a non-trivial share of search impact is not purely organic, but induced by TV exposure.

Overall, the numbers suggest a clear mechanism: TV generates demand (especially Paid TV), and search captures and converts that demand, with RAI further strengthening conversion efficiency.

In [ ]:
coef_benchmark = model_benchmark.params['Search_sat']
coef_2stage    = model_2stage.params['Search_fitted_sat']
tv_induced     = coef_benchmark - coef_2stage
tv_induced_share = tv_induced / coef_benchmark
print(f"TV-induced share of Search coefficient: {tv_induced_share:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

coef_benchmark = model_benchmark.params['Search_sat']
coef_2stage    = model_2stage.params['Search_fitted_sat']
tv_induced     = coef_benchmark - coef_2stage

err_benchmark  = 1.96 * model_benchmark.bse['Search_sat']
err_2stage     = 1.96 * model_2stage.bse['Search_fitted_sat']

# Stacked bars
ax.bar(['Benchmark'], [coef_2stage], color='darkgreen', alpha=0.8, label='Organic search effect')
ax.bar(['Benchmark'], [tv_induced],  color='darkred',    alpha=0.8,
       bottom=[coef_2stage], label='TV-induced portion')
ax.bar(['Two-Stage'], [coef_2stage], color='darkgreen', alpha=0.8)

# CI error bars on top of each full bar
ax.errorbar(['Benchmark'], [coef_benchmark], yerr=[err_benchmark],
            fmt='none', ecolor='black', elinewidth=1.5, capsize=5)
ax.errorbar(['Two-Stage'], [coef_2stage],    yerr=[err_2stage],
            fmt='none', ecolor='black', elinewidth=1.5, capsize=5)

# Annotations inside bars
ax.text(0, coef_2stage / 2,            f'{coef_2stage:.3f}', ha='left', va='center',
        fontsize=9, color='white', fontweight='bold')
ax.text(0, coef_2stage + tv_induced/2, f'+{tv_induced:.3f}', ha='left', va='center',
        fontsize=9, color='white', fontweight='bold')
ax.text(1, coef_2stage / 2,            f'{coef_2stage:.3f}', ha='left', va='center',
        fontsize=9, color='white', fontweight='bold')

ax.set_ylabel('Search coefficient')
ax.set_title('Decomposing the Search Effect\n~16% is TV-induced', fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7.4 Diminishing Returns (Saturation)

We model diminishing returns using an arctan transformation, which means that increases in media spend have a strong effect at low levels but gradually flatten out as spend grows, reflecting decreasing marginal impact.

The optimized parameters ($\alpha$ for adstock decay and $\beta$ for saturation) suggest that channels differ in both persistence and saturation speed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

channels = {
    'Search' : (best['beta_Search'],
                geometric_adstock(data_l2['SEARCH_spend'],   best['alpha_Search'])),
    'CPC/CPM': (best['beta_CPC'],
                geometric_adstock(data_l2['CPC/CPM_spend'],  best['alpha_CPC'])),
}

for ax, (name, (beta, adstock)) in zip(axes, channels.items()):
    x_max  = adstock.max() * 1.5
    x_vals = np.linspace(0, x_max, 300)
    y_vals = np.arctan(x_vals / beta)

    mean_a  = adstock.mean()
    mean_sat = np.arctan(mean_a / beta)

    ax.plot(x_vals, y_vals, color='steelblue', linewidth=2, label='Saturation curve')
    ax.axvline(mean_a, color='tomato', linestyle='--', linewidth=1.5, label=f'Mean adstock')
    ax.scatter([mean_a], [mean_sat], color='tomato', zorder=5, s=60)

    # shade the "room left" region
    ax.axvspan(mean_a, x_max, alpha=0.08, color='green', label='Room to scale')

    ax.set_xlabel('Adstock value')
    ax.set_ylabel('Saturated value (arctan)')
    ax.set_title(f'{name}  (β = {beta:,.0f})')
    ax.legend(fontsize=8)

fig.suptitle('Saturation Curves: Where Do We Currently Operate?', fontweight='bold')
plt.tight_layout()
plt.show()

**Search and CPC** were modeled with both adstock and arctan saturation.
- Both decay quickly ($\alpha = 0.1$), consistent with intent-driven behaviour
where effects dissipate within days.
- Their saturation thresholds differ
substantially: Search ($\beta ≈ 668$) is already operating in the diminishing
returns zone, meaning additional investment yields smaller incremental gains.
- CPC ($\beta ≈ 11,222$) remains on the steeper part of the curve,
suggesting the marginal euro spent there is still more efficient. If the
budget is fixed, reallocating from Search toward CPC should improve
overall trial volume.

In contrast, **TV and Radio** were modeled with adstock only.
- This is a deliberate choice: GRPs measure audience reach rather than direct spend, and the
sparse, episodic nature of TV campaigns makes saturation difficult to
identify reliably from data alone.
- Their high decay rate (α = 0.7 for TV,
0.5 for Radio) reflects the well-known persistence of TV's brand-building
effect over multiple days.

Overall, the results indicate that performance channels (Search, CPC) saturate quickly and decay quickly, while offline channels (TV) have slower decay and more sustained impact over time.

## 7.5 Creative flights


In [ ]:
flights = ['TV_RAI_adstock1', 'TV_RAI_adstock2', 'TV_RAI_adstock3', 'TV_RAI_adstock4']

flight_df = pd.DataFrame({
    'Flight'      : ['OnAir 1', 'OnAir 2', 'OnAir 3', 'OnAir 4'],
    'Coefficient' : [model_final.params[f] for f in flights],
    'Std Error'   : [model_final.bse[f]    for f in flights],
    'p-value'     : [model_final.pvalues[f] for f in flights],
}).round(4)

def sig_stars(p):
    if p < 0.01:  return '***'
    if p < 0.05:  return '**'
    if p < 0.10:  return '*'
    return '— (p = {:.3f})'.format(p)

flight_df['Significant'] = flight_df['p-value'].apply(sig_stars)
flight_df = flight_df.sort_values('Coefficient', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

flights_plot = flight_df.sort_values('Coefficient', ascending=True)

ax.barh(
    flights_plot['Flight'],
    flights_plot['Coefficient'],
    xerr=1.96 * flights_plot['Std Error'],  # 95% confidence interval
    color='lightcoral',
    alpha=0.8,
    capsize=4,
    edgecolor='white'
)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (effect on log Trials per GRP)')
ax.set_title('RAI Creative Flight Effectiveness\nwith 95% Confidence Intervals')

# Annotate with significance stars
for i, row in flights_plot.reset_index(drop=True).iterrows():
    ax.text(
        row['Coefficient'] + 1.96 * row['Std Error'] + 0.001,
        i,
        row['Significant'],
        va='center', fontsize=10, color='lightcoral'
    )

plt.tight_layout()
plt.show()

All four flights were statistically significant, meaning each creative
generated a measurable and reliable lift in app trials.

- **OnAir 3 was the strongest performer** — both the highest coefficient
and the most precisely estimated (smallest standard error), suggesting
a consistent and well-identified effect. This creative should be the
reference point for future campaign planning.

- **OnAir 1, while significant, carries more uncertainty** than the others — its standard error is more than double that of Flight 3. This
likely reflects the fact that it aired earliest in the campaign, when
the product was less established and the sample of TV-on days available
to identify its effect was smaller.

- **OnAir 2 and OnAir 4 are statistically indistinguishable** from each
other (0.0351 vs 0.0349), both performing solidly but below Flight 3.

## 7.6 Elasticity Computation

In this section, we compute channel elasticities to make media effects comparable on a common scale.

An advertising elasticity answers the following question: if spending on a channel increases by 1%, by how much do app trials increase, in percentage terms?

This is more informative than raw coefficients because the final model is specified in log-level form and includes adstock and nonlinear saturation. As a result, the regression coefficient alone is not an elasticity.



**For channels with both adstock and saturation**, we evaluate elasticity at the sample mean using the derivative of the saturation function.

In our specification: $
\text{sat}(A) = \arctan\left(\frac{A}{\beta}\right)
$

so its derivative is: $
\frac{d\,\text{sat}(A)}{dA} = \frac{\frac1\beta}{1 + (\frac A\beta)^2}
$

For these channels,

- short-run elasticity is computed as: $
\varepsilon^{SR} = b \cdot \text{sat}'(\bar A) \cdot \bar S$

- long-run elasticity is computed as:  $
\varepsilon^{LR} = \frac{b \cdot \text{sat}'(\bar A) \cdot \bar S}{1-\alpha}
$

where:
- $b$ is the estimated coefficient from the final model  
- $\bar{A}$ is mean adstock  
- $\bar{S}$ is mean spend  
- $\alpha$ is the adstock decay parameter  

**For adstock-only channels** (TV and Radio), the calculation is simpler since saturation is not considered:

- short-run elasticity: $ \varepsilon^{SR} = b \cdot \bar A$
- long-run elasticity:  $\varepsilon^{LR} = \frac{b \cdot \bar A}{1-\alpha}$

where $\bar A$ is the mean of the **adstocked** media variable.

In [ ]:
# Part 7.6 Helper functions
def atan_derivative(adstock_value, beta):
    return (1 / beta) / (1 + (adstock_value / beta) ** 2)

def elasticity_sat_loglevel(model, data, coef_name, spend_col, adstock_col, alpha, beta):
    coef = model.params[coef_name]
    mean_spend = data[spend_col].mean()
    mean_adstock = data[adstock_col].mean()

    sat_slope = atan_derivative(mean_adstock, beta)

    short_run = coef * sat_slope * mean_spend
    long_run = short_run / (1 - alpha)

    return {
        "coef": coef,
        "mean_spend": mean_spend,
        "mean_adstock": mean_adstock,
        "sat_slope_at_mean": sat_slope,
        "elasticity_short_run": short_run,
        "elasticity_long_run": long_run
    }


def elasticity_adstock_loglevel(model, data, coef_name, input_col, alpha):
    coef = model.params[coef_name]
    mean_input = data[input_col].mean()

    short_run = coef * mean_input
    long_run = short_run / (1 - alpha)

    return {
        "coef": coef,
        "mean_input": mean_input,
        "elasticity_short_run": short_run,
        "elasticity_long_run": long_run
    }

### 7.6.1 Parameter choices

For Search and CPC, we use the parameters selected by grid search in Section 4.2.

- Search: $\alpha = 0.1$, $\beta \approx 668.47$  
- CPC: $\alpha = 0.1$, $\beta \approx 11221.95$  

For TV and Radio, saturation was not estimated in the final model, so we compute elasticities using the adstock-only specification.

- For RAI TV we use $\alpha = 0.7$, consistent with the feature construction used in the notebook
- For Radio we use $\alpha = 0.5$.

In [ ]:
# Parameters selected earlier in the notebook
alpha_search = best['alpha_Search']
alpha_cpc = best['alpha_CPC']
beta_search = best['beta_Search']
beta_cpc = best['beta_CPC']

# Assumed / fixed decay values used for adstock-only channels
alpha_tv_rai = 0.7
alpha_radio = 0.5

### 7.6.2 Channel-level elasticity calculations

We first compute elasticities for digital channels with both adstock and saturation, and then for offline channels modeled with adstock only.

In [ ]:
# Search elasticity
search_elas = elasticity_sat_loglevel(
    model=model_final,
    data=data_l2,
    coef_name="Search_sat",
    spend_col="SEARCH_spend",
    adstock_col="Search_adstock",
    alpha=alpha_search,
    beta=beta_search)

# CPC elasticity
cpc_elas = elasticity_sat_loglevel(
    model=model_final,
    data=data_l2,
    coef_name="CPC_sat",
    spend_col="CPC/CPM_spend",
    adstock_col="CPC_adstock",
    alpha=alpha_cpc,
    beta=beta_cpc)


In [ ]:
# RAI TV flights
tv1_elas = elasticity_adstock_loglevel(
    model=model_final,
    data=data_l2,
    coef_name="TV_RAI_adstock1",
    input_col="TV_Grp_Rai3_OnAir1",
    alpha=alpha_tv_rai
)

tv2_elas = elasticity_adstock_loglevel(
    model=model_final,
    data=data_l2,
    coef_name="TV_RAI_adstock2",
    input_col="TV_Grp_Rai3_OnAir2",
    alpha=alpha_tv_rai
)

tv3_elas = elasticity_adstock_loglevel(
    model=model_final,
    data=data_l2,
    coef_name="TV_RAI_adstock3",
    input_col="TV_Grp_Rai3_OnAir3",
    alpha=alpha_tv_rai
)

tv4_elas = elasticity_adstock_loglevel(
    model=model_final,
    data=data_l2,
    coef_name="TV_RAI_adstock4",
    input_col="TV_Grp_Rai3_OnAir4",
    alpha=alpha_tv_rai
)

# Paid TV
tv_paid_elas = elasticity_adstock_loglevel(
    model=model_final,
    data=data_l2,
    coef_name="TV_Paid_adstock",
    input_col="TV_Grp_Mediaset_Premium_Sky",
    alpha=alpha_tv_rai
)

# Radio
radio_elas = elasticity_adstock_loglevel(
    model=model_final,
    data=data_l2,
    coef_name="Radio_adstock",
    input_col="RADIO_Grp",
    alpha=alpha_radio
)


### 7.6.3 Interpretation / Summary

In [ ]:
sr_elasticity_search   = search_elas['elasticity_short_run']
sr_elasticity_tv_direct = tv_paid_elas['elasticity_short_run']

# Indirect path: TV-induced share of Search elasticity
sr_elasticity_tv_indirect = tv_induced_share * sr_elasticity_search
sr_elasticity_tv_total    = sr_elasticity_tv_direct + sr_elasticity_tv_indirect
true_multiplier           = sr_elasticity_tv_total / sr_elasticity_tv_direct

print(f"Direct TV SR elasticity:                {sr_elasticity_tv_direct:.4f}")
print(f"Indirect TV SR elasticity (via Search): {sr_elasticity_tv_indirect:.4f}")
print(f"Total TV SR elasticity:                 {sr_elasticity_tv_total:.4f}")
print(f"True contribution multiple:             {true_multiplier:.1f}×")

In [ ]:
elasticity_table = pd.DataFrame([
    {
        "Channel": "Search",
        "Type": "adstock + saturation",
        "Short-run elasticity": search_elas["elasticity_short_run"],
        "Long-run elasticity": search_elas["elasticity_long_run"]
    },
    {
        "Channel": "CPC",
        "Type": "adstock + saturation",
        "Short-run elasticity": cpc_elas["elasticity_short_run"],
        "Long-run elasticity": cpc_elas["elasticity_long_run"]
    },
    {
        "Channel": "Radio",
        "Type": "adstock only",
        "Short-run elasticity": radio_elas["elasticity_short_run"],
        "Long-run elasticity": radio_elas["elasticity_long_run"]
    },
    {
        "Channel": "TV_Paid",
        "Type": "adstock only",
        "Short-run elasticity": tv_paid_elas["elasticity_short_run"],
        "Long-run elasticity": tv_paid_elas["elasticity_long_run"]
    },
    {
        "Channel": "TV_RAI_flight3",
        "Type": "adstock only",
        "Short-run elasticity": tv3_elas["elasticity_short_run"],
        "Long-run elasticity": tv3_elas["elasticity_long_run"]
    },
    {
        "Channel": "TV_RAI_flight4",
        "Type": "adstock only",
        "Short-run elasticity": tv4_elas["elasticity_short_run"],
        "Long-run elasticity": tv4_elas["elasticity_long_run"]
    },
    {
        "Channel": "TV_RAI_flight2",
        "Type": "adstock only",
        "Short-run elasticity": tv2_elas["elasticity_short_run"],
        "Long-run elasticity": tv2_elas["elasticity_long_run"]
    },
    {
        "Channel": "TV_RAI_flight1",
        "Type": "adstock only",
        "Short-run elasticity": tv1_elas["elasticity_short_run"],
        "Long-run elasticity": tv1_elas["elasticity_long_run"]
    }
])

elasticity_table = elasticity_table.sort_values("Short-run elasticity", ascending=False)
elasticity_table.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

channels = elasticity_table['Channel']
sr = elasticity_table['Short-run elasticity']
lr = elasticity_table['Long-run elasticity']

x     = np.arange(len(channels))
width = 0.35

ax.barh(x + width/2, lr, width, label='Long-run',  color='midnightblue', alpha=0.8)
ax.barh(x - width/2, sr, width, label='Short-run', color='lightsteelblue',    alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(channels, fontsize=9)
ax.set_xlabel('Elasticity')
ax.set_title('Advertising Elasticities by Channel\n(Short-run vs Long-run)', fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
ax.legend(fontsize=9)

# Light divider between digital and offline
digital_cutoff = list(channels).index('Radio') - 0.5
ax.axhline(digital_cutoff, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.text(ax.get_xlim()[1] * 0.98, digital_cutoff + 0.1, 'digital ↓  offline ↑',
        ha='right', fontsize=7, color='gray')

plt.tight_layout()
plt.show()

Consistent with previous sections, **Search** has the highest elasticity (≈ 0.139 short-run, 0.154 long-run), followed by **CPC** (≈ 0.099 and 0.110), confirming that performance channels are the most effective at the margin.

In contrast, **offline channels** show much lower elasticities. **Radio** performs slightly better than TV (≈ 0.006 short-run), while **TV (RAI and Paid)** remains low in the short run (≈ 0.001–0.005) but increases in the long run (up to ≈ 0.016) due to carry-over.

Overall, this aligns with earlier findings: digital channels drive immediate conversions, while TV and Radio play a supporting, demand-generation role with effects that build over time.

**Caveats**

OLS elasticities are intentionally conservative — AR lags control for baseline trial momentum, reducing the risk of attributing organic growth to media spend. With observational data, media activity naturally coincides with periods of high organic demand. Meridian's posterior means (no AR lags by design) represent the upper bound.
The true channel effect likely sits between the two.

# 8. Business Takeaway/Actionable insights

## Offline: demand generation

**Don't judge TV Paid by its direct effect alone**
> About 16% of Search-driven trials trace back to TV. Once the indirect path
is included, TV Paid's true contribution is roughly 5× its face-value
elasticity. Factor this in before cutting TV Paid budget.

**Keep all four RAI flights — don't cherry-pick**

> All flights are significant and broadly comparable. No creative clearly
outperforms the others, so concentration on specific flights is not
warranted by the data.

**Radio for short campaigns, TV Paid for long ones**

> Radio's short-run efficiency edges TV Paid. But TV Paid's slower decay
means it accumulates more value over time. Match the channel to the
campaign horizon.


## Digital: demand capture

**Grow Search quality, not Search volume**
> Search is saturated at the aggregate level — more budget won't yield proportional returns. The priority is concentrating spend in high-conversion
windows rather than scaling overall.

**Maintain CPC - it is earning its allocation**
> Strong second channel with room left on the saturation curve.
Hold current levels.

**Cut CPD**
> Weak and unstable across specifications. Redeploy that budget to
Search Always-On or CPC.


## Cross-channel

**TV and Search must be planned together**

> - TV-active days double search click volume.
> - Search running outside a TV flight converts at a fundamentally lower rate.
>
> Align Search pacing with
TV scheduling: pulling back Search during flights or maintaining full
Search budget in dark periods both leave value on the table.